# AC-PFL — FD002 Experiments

Additional experiment runs for the FD002 C-MAPSS subset (including extra seeds 404-808 for AC-PFL, FedAvg, and FedProx).

> **Note:** this notebook was run against an earlier snapshot of the core code (before the condition-subset filtering feature was added to `utils.py` / `run_experiment.py`). It's archived here for its unique FD002 runs; for the current code, see [`../src/`](../src/).

Core algorithm code lives in [`../src/`](../src/) — this notebook only contains the calls that produced the logged results in [`../results/`](../results/).


In [ ]:
# Core modules live in ../src/ (config.py, utils.py, preprocess.py, run_experiment.py)
# On Kaggle/Colab, copy src/*.py into the working directory first, e.g.:
# import shutil; [shutil.copy(f'../src/{f}', '.') for f in
#     ['config.py', 'utils.py', 'preprocess.py', 'run_experiment.py']]


In [5]:
import os

# Lock the GPUs before TensorFlow wakes up
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

In [ ]:
import itertools
import os
import pandas as pd
import traceback

OUTPUT = '/kaggle/working/results.csv'

def already_completed(output_file, method, dataset, seed):
    if not os.path.exists(output_file):
        return False
    try:
        df = pd.read_csv(output_file)
        return any(
            (df['method'] == method) &
            (df['dataset'] == dataset) &
            (df['seed'] == seed)
        )
    except Exception:
        return False

def save_result(result, output_file):
    exists = os.path.exists(output_file)
    import csv
    with open(output_file, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=result.keys())
        if not exists:
            w.writeheader()
        w.writerow(result)
    print(f"  ✅ Saved: {result['method']} {result['dataset']} "
          f"seed {result['seed']} | "
          f"MAE={result['test_mae']} | NASA={result['nasa_score']}")

from run_experiment import run_simulation, run_cfl, run_acpfl

METHODS = ['fedavg', 'fedprox', 'fedper', 'ditto', 'cfl', 'acpfl']
DATASETS = ['FD001', 'FD002', 'FD003', 'FD004']
SEEDS = [42, 101, 2026]

# Assign methods to accounts to parallelize
# Account 1: fedavg, fedprox
# Account 2: fedper, ditto  
# Account 3: cfl
# Account 4: acpfl

# Change this per account
METHODS_THIS_ACCOUNT = ['fedavg', 'fedprox']  # CHANGE PER ACCOUNT

total = len(METHODS_THIS_ACCOUNT) * len(DATASETS) * len(SEEDS)
completed = 0
skipped = 0
failed = 0

print(f"Total planned runs: {total}")
print(f"Methods: {METHODS_THIS_ACCOUNT}")

for method, dataset, seed in itertools.product(
    METHODS_THIS_ACCOUNT, DATASETS, SEEDS
):
    if already_completed(OUTPUT, method, dataset, seed):
        print(f"  ⏭️  Skip: {method} {dataset} seed {seed}")
        skipped += 1
        continue

    print(f"\n{'='*50}")
    print(f"  Running: {method} | {dataset} | seed {seed}")
    print(f"  Progress: {completed} done, {skipped} skipped, {failed} failed")
    print(f"{'='*50}")

    try:
        if method == 'cfl':
            result = run_cfl(dataset, seed)
        elif method == 'acpfl':
            result = run_acpfl(dataset, seed, alpha=0.5)
        else:
            result = run_simulation(method, dataset, seed)

        save_result(result, OUTPUT)
        completed += 1

    except Exception as e:
        print(f"  ❌ FAILED: {method} {dataset} seed {seed}")
        print(f"  Error: {e}")
        traceback.print_exc()
        failed += 1
        continue

print(f"\n{'='*50}")
print(f"DONE. Completed: {completed} | Skipped: {skipped} | Failed: {failed}")

In [9]:
from run_experiment import run_simulation
print("Checking FedAvg seed 101...")
result = run_simulation('fedavg', 'FD004', 101)
print("FedAvg 101:", result)

Checking FedAvg seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


INFO flwr 2026-06-29 12:08:59,416 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-06-29 12:09:08,244	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-06-29 12:09:11,549 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'memory': 18502853018.0, 'object_store_memory': 7929794150.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-06-29 12:09:11,550 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-06-29 12:09:11,651 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-06-29 12:09:11,652 | server.py:89 | Initializing global parameters
INFO flwr 2026-06-29 12:09:11,654 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-06-29 12:09:11,655 | server.py:91 | Evaluating initial parameters
(pid=31983) WARNING: All

  [Round 0] Test MAE: 78.6153 | NASA: 1644382.07


(DefaultActor pid=31984) I0000 00:00:1782734962.017995   31984 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=31984) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=31984) E0000 00:00:1782734952.773077   31984 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=31984) E0000 00:00:1782734952.797318   31984 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 34.6112 | NASA: 12939.21


...

  [Round 49] Test MAE: 21.9383 | NASA: 16324.28


DEBUG flwr 2026-06-29 12:58:55,348 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-06-29 12:58:55,349 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-06-29 12:59:43,637 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 12:59:48,679 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.38461628267842, 'nasa_score': 12846.124463191962}, 3029.6043184459995)
DEBUG flwr 2026-06-29 12:59:48,680 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.3846 | NASA: 12846.12


DEBUG flwr 2026-06-29 12:59:52,220 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 12:59:52,221 | server.py:153 | FL finished in 3033.146371336001
INFO flwr 2026-06-29 12:59:52,222 | app.py:225 | app_fit: losses_distributed [(1, 2110.3554153560954), (2, 582.3708549266322), (3, 503.0775718998567), (4, 505.333132544076), (5, 520.8408760989068), (6, 533.2610897432438), (7, 622.6179469613662), (8, 540.0756214677392), (9, 621.449569562498), (10, 564.5430481452395), (11, 584.4884499688363), (12, 603.0074242084802), (13, 569.4978849437031), (14, 590.3279573193199), (15, 598.5299924133437), (16, 587.7251478144798), (17, 637.1615664719753), (18, 581.8152727520006), (19, 613.297999001768), (20, 606.015488606293), (21, 678.1334789699686), (22, 633.8578895196007), (23, 635.6229544527439), (24, 641.7534628940399), (25, 654.1987102694965), (26, 625.7253406638619), (27, 674.200442593943), (28, 630.1799470074455), (29, 630.1054458947528), (30, 620.75927945168

FedAvg 101: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 101, 'test_mae': 22.3846, 'nasa_score': 12846.12, 'comm_kb': 28900.78}


In [8]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 101...")
result = run_acpfl('FD004', 101, alpha=0.5)
print("AC-PFL 101:", result)

Checking AC-PFL seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


I0000 00:00:1782728957.816347     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782728957.822175     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782728965.445161     165 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [76419.0, 104342.7, 102440.1, 37814.8]
  [Round 2] Val NASA per client: [60500.2, 131413.1, 38912.4, 30253.3]
  [Round 3] Val NASA per client: [60125.7, 73950.0, 32040.1, 33713.2]
  [Round 4] Val NASA per client: [129263.5, 65683.5, 37613.8, 30054.7]
  [Round 5] Re-clustered (α=1.0). Changes: {1: (0, 1), 2: (1, 0)}
  [Round 5] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 5] Cluster 0 val NASA: 69654.28
  [Round 5] Cluster 1 val NASA: 48264.89
  [Round 5] Val NASA per client: [87740.9, 63574.2, 51567.7, 32955.6]
  [Round 6] Val NASA per client: [61567.2, 74104.3, 31157.7, 29151.1]
  [Round 7] Val NASA per client: [133787.5, 82914.6, 30320.7, 36528.0]
  [Round 8] Val NASA per client: [48399.7, 80176.4, 51081.3, 30661.1]
  [Round 9] Val NASA per client: [60160.4, 92043.0, 38400.3, 29203.1]
  [Round 10] Re-clustered (α=0.5). Changes: {1: (1, 0), 2: (0, 1)}
  [Round 10] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 10] Cluster 0 val NASA: 

In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 42...")
result = run_acpfl('FD004', 42, alpha=0.5)
print("AC-PFL 42:", result)

E0000 00:00:1782750267.857125     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782750267.981688     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782750268.942533     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942584     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942587     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942589     112 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


I0000 00:00:1782750339.389640     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782750339.395700     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782750347.340774     180 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [139898.1, 49667.5, 41624.7, 32652.3]
  [Round 2] Val NASA per client: [143416.2, 95197.9, 41664.9, 17370.7]
  [Round 3] Val NASA per client: [176482.2, 63069.0, 32794.7, 12313.4]
  [Round 4] Val NASA per client: [307796.9, 93731.4, 24326.9, 12635.3]
  ⚠️ Spectral produced invalid clusters {0: 1, 1: 3}. Falling back to KMeans.
  KMeans fallback result: {1: 1, 0: 3}
  [Round 5] Re-clustered (α=1.0). Changes: {0: (0, 1), 2: (1, 0), 3: (1, 0)}
  [Round 5] Assignments: {'0': 1, '1': 0, '2': 0, '3': 0}
  [Round 5] Cluster 0 val NASA: 48300.18
  [Round 5] Cluster 1 val NASA: 209179.38
  [Round 5] Val NASA per client: [209179.4, 99070.2, 30381.2, 15449.1]
  [Round 6] Val NASA per client: [39358.2, 126635.7, 27570.5, 25448.9]
  [Round 7] Val NASA per client: [120220.2, 87195.0, 27480.0, 18245.4]
  [Round 8] Val NASA per client: [68654.7, 71723.1, 34401.3, 19520.5]
  [Round 9] Val NASA per client: [55398.2, 136375.9, 30807.5, 25683.1]
  [Round 10] Re-clustered (

In [7]:
from run_experiment import run_simulation
print("Checking FedAvg seed 42...")
result = run_simulation('fedavg', 'FD004', 42)
print("FedAvg 42:", result)

Checking FedAvg seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


INFO flwr 2026-06-29 18:15:39,074 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-06-29 18:15:48,409	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-06-29 18:15:52,232 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'memory': 18396964455.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 7884413337.0, 'CPU': 4.0}
INFO flwr 2026-06-29 18:15:52,233 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-06-29 18:15:52,333 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-06-29 18:15:52,334 | server.py:89 | Initializing global parameters
INFO flwr 2026-06-29 18:15:52,336 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-06-29 18:15:52,336 | server.py:91 | Evaluating initial parameters
(pid=33157) WARNING: All

  [Round 0] Test MAE: 80.3084 | NASA: 1874047.15


(DefaultActor pid=33157) I0000 00:00:1782756963.311607   33157 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13374 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=33156) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=33156) E0000 00:00:1782756953.511946   33156 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=33156) E0000 00:00:1782756953.541538   33156 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 39.0170 | NASA: 73795.66


...

  [Round 49] Test MAE: 21.1101 | NASA: 17697.67


DEBUG flwr 2026-06-29 19:10:19,004 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-06-29 19:10:19,005 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-06-29 19:11:15,365 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 19:11:20,694 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.556122179954283, 'nasa_score': 31440.990196730054}, 3320.5397188999996)
DEBUG flwr 2026-06-29 19:11:20,695 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.5561 | NASA: 31440.99


DEBUG flwr 2026-06-29 19:11:23,265 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 19:11:23,266 | server.py:153 | FL finished in 3323.1120386559996
INFO flwr 2026-06-29 19:11:23,268 | app.py:225 | app_fit: losses_distributed [(1, 1888.2686313791535), (2, 740.4444458646602), (3, 712.6677854725493), (4, 574.3667632697321), (5, 574.737801039616), (6, 561.780703564291), (7, 568.9570834250161), (8, 589.1928111457652), (9, 632.4505906743735), (10, 607.16072449056), (11, 650.7888930670941), (12, 645.4695448451711), (13, 658.5004925439448), (14, 691.6299695958071), (15, 691.9236625432608), (16, 743.6538397409048), (17, 725.5946467260342), (18, 724.0221167156799), (19, 696.5653986570472), (20, 715.1448718172043), (21, 719.6221619604387), (22, 723.6485659459281), (23, 724.2454473006602), (24, 741.4719130301171), (25, 718.7479872955479), (26, 725.2752653853063), (27, 762.5117739316287), (28, 741.4240322061212), (29, 786.57879630687), (30, 740.56120001320

FedAvg 42: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 42, 'test_mae': 21.5561, 'nasa_score': 31440.99, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 2026...")
result = run_acpfl('FD004', 2026, alpha=0.5)
print("AC-PFL 2026:", result)

E0000 00:00:1782806943.769476     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782806943.836301     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782806944.373266     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373320     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373323     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373325     114 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)


I0000 00:00:1782807007.984322     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782807007.990197     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782807015.277449     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [248610.6, 62376.2, 50338.8, 59521.0]
  [Round 2] Val NASA per client: [21423.7, 83794.0, 42785.2, 89356.1]
  [Round 3] Val NASA per client: [26814.3, 79425.2, 36295.0, 47739.7]
  [Round 4] Val NASA per client: [19332.5, 55592.1, 33722.5, 49475.0]
  [Round 5] Re-clustered (α=1.0). Changes: {1: (0, 1), 3: (1, 0)}
  [Round 5] Assignments: {'0': 0, '1': 1, '2': 1, '3': 0}
  [Round 5] Cluster 0 val NASA: 36741.42
  [Round 5] Cluster 1 val NASA: 51691.86
  [Round 5] Val NASA per client: [20855.3, 60652.8, 42730.9, 52627.5]
  [Round 6] Val NASA per client: [24193.2, 70768.5, 35645.9, 31192.6]
  [Round 7] Val NASA per client: [27469.9, 64258.1, 40833.1, 49291.4]
  [Round 8] Val NASA per client: [19390.7, 90996.0, 35130.0, 36668.9]
  [Round 9] Val NASA per client: [15231.5, 60834.7, 45032.8, 41670.0]
  [Round 10] Re-clustered (α=0.5). Changes: {2: (1, 0), 3: (0, 1)}
  [Round 10] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 10] Cluster 0 val NASA: 5162

In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 202...")
result = run_simulation('fedavg', 'FD004', 202)
print("FedAvg 202:", result)

E0000 00:00:1783061413.014009     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783061413.081289     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783061413.597749     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597800     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597802     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597805     112 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


I0000 00:00:1783061475.680673     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783061475.686557     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-03 06:51:17,556 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-03 06:51:25,606	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-03 06:51:28,942 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 21526421504.0, 'object_store_memory': 9225609216.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-03 06:51:28,943 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 79.5096 | NASA: 1762216.56


(DefaultActor pid=362) I0000 00:00:1783061499.903469     362 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13634 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=361) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=361) E0000 00:00:1783061490.363995     361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=361) E0000 00:00:1783061490.377519     361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 22.3185 | NASA: 70375.75


DEBUG flwr 2026-07-03 07:41:29,107 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-03 07:41:29,108 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-03 07:42:30,833 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 07:42:32,150 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.7314583716854, 'nasa_score': 94065.3112069071}, 3059.460326807)
DEBUG flwr 2026-07-03 07:42:32,151 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.7315 | NASA: 94065.31


DEBUG flwr 2026-07-03 07:42:34,537 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 07:42:34,538 | server.py:153 | FL finished in 3061.848423547
INFO flwr 2026-07-03 07:42:34,539 | app.py:225 | app_fit: losses_distributed [(1, 1978.5757001936547), (2, 707.5090152448369), (3, 606.6025778512512), (4, 593.8671045436449), (5, 554.3076220368134), (6, 555.7166692330135), (7, 568.347766365388), (8, 585.4292547397908), (9, 598.8238107889205), (10, 678.0012805122827), (11, 633.0437550985422), (12, 642.21263172059), (13, 676.6232511390159), (14, 731.246148430685), (15, 701.0002635148551), (16, 707.7408619826356), (17, 701.6863767781468), (18, 742.1739606594689), (19, 740.4893655334496), (20, 744.5630167900324), (21, 752.3963596569912), (22, 774.5815005250266), (23, 739.188650343473), (24, 750.283804180846), (25, 749.726894964321), (26, 759.6116933747086), (27, 747.988471250991), (28, 734.5217870412256), (29, 764.4190314144334), (30, 748.9624733883495), (

FedAvg 202: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 202, 'test_mae': 22.7315, 'nasa_score': 94065.31, 'comm_kb': 28900.78}


In [ ]:
from run_experiment import run_simulation
print("Checking FedAvg seed 303...")
result = run_simulation('fedavg', 'FD004', 303)
print("FedAvg 303:", result)

Checking FedAvg seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-03 08:35:47,413 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-03 08:35:59,415	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-03 08:36:03,050 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6464505446.0, 'CPU': 4.0, 'memory': 15083846042.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-03 08:36:03,052 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-03 08:36:03,082 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-03 08:36:03,087 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-03 08:36:03,090 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-03 08:36:03,091 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-03 08:

  [Round 0] Test MAE: 79.3424 | NASA: 1739396.33


(pid=110315) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=110315) E0000 00:00:1783067765.731620  110315 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=110315) E0000 00:00:1783067765.764978  110315 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=110315) W0000 00:00:1783067765.931578  110315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=110315) W0000 00:00:1783067765.931635  110315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=110315) W0000 00:00:1783067765.931643  110315 computation_placer.cc:177] computation placer already registered. Please check linkage and avo

  [Round 1] Test MAE: 35.8306 | NASA: 97632.96


...

DEBUG flwr 2026-07-03 08:50:56,192 | server.py:187 | evaluate_round 14 received 4 results and 0 failures
DEBUG flwr 2026-07-03 08:50:56,193 | server.py:222 | fit_round 15: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-03 08:51:41,721 | server.py:236 | fit_round 15 received 4 results and 0 failures
INFO flwr 2026-07-03 08:51:43,013 | server.py:125 | fit progress: (15, 0.0, {'mae': 21.4336029483426, 'nasa_score': 19722.619984273966}, 937.9544266820003)
DEBUG flwr 2026-07-03 08:51:43,014 | server.py:173 | evaluate_round 15: strategy sampled 4 clients (out of 4)


  [Round 15] Test MAE: 21.4336 | NASA: 19722.62


DEBUG flwr 2026-07-03 08:51:45,456 | server.py:187 | evaluate_round 15 received 4 results and 0 failures
DEBUG flwr 2026-07-03 08:51:45,457 | server.py:222 | fit_round 16: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-03 08:52:29,016 | server.py:236 | fit_round 16 received 4 results and 0 failures
INFO flwr 2026-07-03 08:52:30,386 | server.py:125 | fit progress: (16, 0.0, {'mae': 20.737856415010267, 'nasa_score': 25748.60741122064}, 985.3277911720006)
DEBUG flwr 2026-07-03 08:52:30,388 | server.py:173 | evaluate_round 16: strategy sampled 4 clients (out of 4)


  [Round 16] Test MAE: 20.7379 | NASA: 25748.61


DEBUG flwr 2026-07-03 08:52:33,305 | server.py:187 | evaluate_round 16 received 4 results and 0 failures
DEBUG flwr 2026-07-03 08:52:33,306 | server.py:222 | fit_round 17: strategy sampled 4 clients (out of 4)


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 101...")
result = run_simulation('fedavg', 'FD002', 101)
print("FedAvg 101:", result)

E0000 00:00:1783965269.426799     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783965269.493853     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783965270.029958     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783965270.030001     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783965270.030004     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783965270.030007     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)


I0000 00:00:1783965330.820053     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783965330.826177     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-13 17:55:32,787 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 17:55:41,097	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 17:55:44,772 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'memory': 21568084788.0, 'GPU': 2.0, 'node:__internal_head__': 1.0, 'object_store_memory': 9243464908.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-13 17:55:44,773 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 74.8939 | NASA: 1545397.21


(DefaultActor pid=432) I0000 00:00:1783965355.722155     432 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=431) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=431) E0000 00:00:1783965346.176888     431 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=431) E0000 00:00:1783965346.215755     431 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 21.6807 | NASA: 16227.14


DEBUG flwr 2026-07-13 18:45:09,562 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 18:45:09,563 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 18:46:11,023 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 18:46:12,537 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.414180254844165, 'nasa_score': 15054.148622572015}, 3023.8632343739996)
DEBUG flwr 2026-07-13 18:46:12,538 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.4142 | NASA: 15054.15


DEBUG flwr 2026-07-13 18:46:15,123 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 18:46:15,123 | server.py:153 | FL finished in 3026.4499389039997
INFO flwr 2026-07-13 18:46:15,124 | app.py:225 | app_fit: losses_distributed [(1, 1958.7926570427815), (2, 668.3741881005061), (3, 670.8572643869226), (4, 721.9021897946902), (5, 621.4904136774347), (6, 605.3780866575669), (7, 659.8657712848927), (8, 683.8267791764134), (9, 687.1052342488757), (10, 766.1700931659673), (11, 749.4072883054968), (12, 749.9307814546303), (13, 760.0357767429865), (14, 784.5387649753311), (15, 811.7963433152572), (16, 809.8095549201864), (17, 783.6959832239227), (18, 802.4633583209611), (19, 824.2573725771263), (20, 818.4631173591835), (21, 796.4990396470475), (22, 817.4139355327181), (23, 815.8561313337568), (24, 810.8376331512338), (25, 777.6106868150678), (26, 763.5988124109408), (27, 780.5368574545644), (28, 800.8013476189275), (29, 785.6584539321956), (30, 787.00267

FedAvg 101: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 101, 'test_mae': 21.4142, 'nasa_score': 15054.15, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_simulation
print("Checking FedAvg seed 202...")
result = run_simulation('fedavg', 'FD002', 202)
print("FedAvg 202:", result)

Checking FedAvg seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8702, 30, 24), y shape = (8702,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (8994, 30, 24), y shape = (8994,)
✅ Created sequences: X shape = (2414, 30, 24), y shape = (2414,)
✅ Created sequences: X shape = (9462, 30, 24), y shape = (9462,)
✅ Created sequences: X shape = (2533, 30, 24), y shape = (2533,)
✅ Created sequences: X shape = (9248, 30, 24), y shape = (9248,)
✅ Created sequences: X shape = (2418, 30, 24), y shape = (2418,)


INFO flwr 2026-07-13 18:47:04,991 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 18:47:14,333	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 18:47:18,495 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 9170980454.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'memory': 21398954394.0, 'GPU': 2.0}
INFO flwr 2026-07-13 18:47:18,496 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 18:47:18,520 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 18:47:18,521 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 18:47:18,522 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 18:47:18,524 | server.py:91 | Evaluating initial parameters
(pid=53853) WARNING: All

  [Round 0] Test MAE: 76.1105 | NASA: 1699036.62


(DefaultActor pid=53853) I0000 00:00:1783968449.965501   53853 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13634 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=53851) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=53851) E0000 00:00:1783968439.855311   53851 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=53852) E0000 00:00:1783968439.968499   53852 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=53852) W0000 00:00:1783968440.036551   53852 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once. [rep

  [Round 1] Test MAE: 28.1658 | NASA: 7459.20


...

  [Round 49] Test MAE: 19.5303 | NASA: 18274.80


DEBUG flwr 2026-07-13 19:34:23,194 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 19:34:23,196 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 19:35:23,591 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 19:35:25,124 | server.py:125 | fit progress: (50, 0.0, {'mae': 19.53160320561825, 'nasa_score': 18843.988822427582}, 2884.229885668)
DEBUG flwr 2026-07-13 19:35:25,125 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 19.5316 | NASA: 18843.99


DEBUG flwr 2026-07-13 19:35:27,830 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 19:35:27,831 | server.py:153 | FL finished in 2886.9368003150003
INFO flwr 2026-07-13 19:35:27,832 | app.py:225 | app_fit: losses_distributed [(1, 1109.0368283508046), (2, 677.6754565564036), (3, 624.6409984197163), (4, 612.8997993185459), (5, 688.8405839777571), (6, 625.203628868135), (7, 676.1754971345508), (8, 617.6325589798769), (9, 677.5967845475529), (10, 788.8561765102634), (11, 674.9698313292555), (12, 699.4352110354747), (13, 727.0399370013689), (14, 688.4358924734329), (15, 829.9465025518206), (16, 711.9439116769229), (17, 722.8972429014863), (18, 715.8372773003605), (19, 770.5609458706423), (20, 711.7523932896731), (21, 719.405854754913), (22, 727.9706815795214), (23, 794.4068367286619), (24, 710.889331860777), (25, 754.6652703778431), (26, 692.320684283902), (27, 742.6449576501974), (28, 749.7261898702017), (29, 699.9431745528592), (30, 751.283772649

FedAvg 202: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 202, 'test_mae': 19.5316, 'nasa_score': 18843.99, 'comm_kb': 28900.78}


In [5]:
from run_experiment import run_simulation
print("Checking FedAvg seed 303...")
result = run_simulation('fedavg', 'FD002', 303)
print("FedAvg 303:", result)

E0000 00:00:1784013876.249976     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784013876.379402     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784013877.522261     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784013877.522309     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784013877.522311     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784013877.522314     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8924, 30, 24), y shape = (8924,)
✅ Created sequences: X shape = (2226, 30, 24), y shape = (2226,)
✅ Created sequences: X shape = (9116, 30, 24), y shape = (9116,)
✅ Created sequences: X shape = (2292, 30, 24), y shape = (2292,)
✅ Created sequences: X shape = (9556, 30, 24), y shape = (9556,)
✅ Created sequences: X shape = (2439, 30, 24), y shape = (2439,)
✅ Created sequences: X shape = (9280, 30, 24), y shape = (9280,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)


I0000 00:00:1784013936.838912     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784013936.844833     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-14 07:25:39,244 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 07:25:47,471	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 07:25:51,049 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 21567588762.0, 'CPU': 4.0, 'object_store_memory': 9243252326.0, 'GPU': 2.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-14 07:25:51,050 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 75.2937 | NASA: 1591056.19


(DefaultActor pid=364) I0000 00:00:1784013961.534678     364 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=365) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=365) E0000 00:00:1784013952.430902     365 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=365) E0000 00:00:1784013952.445402     365 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 18.8314 | NASA: 21589.37


DEBUG flwr 2026-07-14 08:05:51,974 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-14 08:05:51,975 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-14 08:06:31,752 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-14 08:06:33,041 | server.py:125 | fit progress: (50, 0.0, {'mae': 17.91934857313237, 'nasa_score': 13806.765048777135}, 2438.107216122)
DEBUG flwr 2026-07-14 08:06:33,042 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 17.9193 | NASA: 13806.77


DEBUG flwr 2026-07-14 08:06:35,249 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-14 08:06:35,250 | server.py:153 | FL finished in 2440.316729068
INFO flwr 2026-07-14 08:06:35,251 | app.py:225 | app_fit: losses_distributed [(1, 1888.541904466203), (2, 551.4304914681696), (3, 522.3647767354497), (4, 500.41153463000063), (5, 428.3371298903382), (6, 418.458589351523), (7, 431.20882329131354), (8, 474.49739038287146), (9, 475.84892783644216), (10, 508.61418628906983), (11, 479.8879257507553), (12, 481.41193075750647), (13, 468.6828044673736), (14, 488.48254897550305), (15, 519.8588215843151), (16, 565.82068562255), (17, 538.681330837122), (18, 542.0829336465866), (19, 561.6872666742847), (20, 551.0913839551503), (21, 576.4601778563016), (22, 576.60367681321), (23, 606.0816788230911), (24, 600.624096283805), (25, 574.4196999194752), (26, 596.4089074823313), (27, 599.762612863104), (28, 603.3298719002529), (29, 604.0578003386978), (30, 594.88981969915

FedAvg 303: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 303, 'test_mae': 17.9193, 'nasa_score': 13806.77, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 2026...")
result = run_simulation('fedavg', 'FD002', 2026)
print("FedAvg 2026:", result)

Checking FedAvg seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8764, 30, 24), y shape = (8764,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)
✅ Created sequences: X shape = (9013, 30, 24), y shape = (9013,)
✅ Created sequences: X shape = (2395, 30, 24), y shape = (2395,)
✅ Created sequences: X shape = (9164, 30, 24), y shape = (9164,)
✅ Created sequences: X shape = (2831, 30, 24), y shape = (2831,)
✅ Created sequences: X shape = (9304, 30, 24), y shape = (9304,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-14 08:07:14,678 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 08:07:25,207	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 08:07:28,610 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 6551772364.0, 'GPU': 2.0, 'memory': 15287468852.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-14 08:07:28,611 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 08:07:28,641 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 08:07:28,643 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 08:07:28,644 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 08:07:28,644 | server.py:91 | Evaluating initial parameters
(pid=53550) WARNING: All

  [Round 0] Test MAE: 75.3955 | NASA: 1605988.15


(DefaultActor pid=53548) I0000 00:00:1784016462.593704   53548 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=53547) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=53547) E0000 00:00:1784016451.325184   53547 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=53547) E0000 00:00:1784016451.366191   53547 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=53548) W0000 00:00:1784016451.633767   53548 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once. [rep

  [Round 1] Test MAE: 38.7603 | NASA: 60769.30


...

  [Round 49] Test MAE: 18.5821 | NASA: 11001.08


DEBUG flwr 2026-07-14 08:49:33,282 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-14 08:49:33,283 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-14 08:50:10,082 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-14 08:50:11,374 | server.py:125 | fit progress: (50, 0.0, {'mae': 18.198616793717196, 'nasa_score': 11054.495743832413}, 2560.599303057)
DEBUG flwr 2026-07-14 08:50:11,375 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 18.1986 | NASA: 11054.50


DEBUG flwr 2026-07-14 08:50:13,687 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-14 08:50:13,688 | server.py:153 | FL finished in 2562.912827635
INFO flwr 2026-07-14 08:50:13,689 | app.py:225 | app_fit: losses_distributed [(1, 1903.7127081681904), (2, 797.1556167434255), (3, 683.8063369423587), (4, 578.8005015555092), (5, 640.5688389483066), (6, 789.601905181744), (7, 548.0077105595398), (8, 469.8793616821705), (9, 491.5534996122976), (10, 529.8275952994143), (11, 550.0954393716142), (12, 584.1160450765741), (13, 609.4427514956857), (14, 587.4690151792121), (15, 560.0103599620243), (16, 547.0952174261479), (17, 651.8761657506783), (18, 590.2457228699403), (19, 625.4907393806416), (20, 655.1836529676867), (21, 629.3455130862214), (22, 709.3916099032779), (23, 651.3795712296796), (24, 667.429283972802), (25, 706.7337800877497), (26, 744.6940374563709), (27, 691.6260844357439), (28, 706.5961053242826), (29, 682.6745483569781), (30, 665.20275217396

FedAvg 2026: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 2026, 'test_mae': 18.1986, 'nasa_score': 11054.5, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_simulation
print("Running FedPer (cluster-routed)...")
fedper_result = run_simulation('fedper', 'FD002', 42)
print("FedPer:", fedper_result)

Running FedPer (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8805, 30, 24), y shape = (8805,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)
✅ Created sequences: X shape = (9138, 30, 24), y shape = (9138,)
✅ Created sequences: X shape = (2270, 30, 24), y shape = (2270,)
✅ Created sequences: X shape = (9413, 30, 24), y shape = (9413,)
✅ Created sequences: X shape = (2582, 30, 24), y shape = (2582,)
✅ Created sequences: X shape = (9283, 30, 24), y shape = (9283,)
✅ Created sequences: X shape = (2383, 30, 24), y shape = (2383,)


INFO flwr 2026-07-14 08:50:52,674 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 08:51:04,398	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 08:51:07,856 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15127909172.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'object_store_memory': 6483389644.0}
INFO flwr 2026-07-14 08:51:07,857 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 08:51:07,883 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 08:51:07,884 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 08:51:07,885 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 08:51:07,885 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-14 08:

FedPer: {'method': 'fedper', 'dataset': 'FD002', 'seed': 42, 'test_mae': 17.7594, 'nasa_score': 4688.83, 'comm_kb': 27650.0}


In [8]:
from run_experiment import run_simulation
print("Running FedPer (cluster-routed)...")
fedper_result = run_simulation('fedper', 'FD002', 101)
print("FedPer:", fedper_result)

Running FedPer (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)


INFO flwr 2026-07-14 09:30:33,831 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 09:30:44,299	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 09:30:47,813 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'memory': 15186038784.0, 'object_store_memory': 6508302336.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0}
INFO flwr 2026-07-14 09:30:47,814 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 09:30:47,842 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 09:30:47,849 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 09:30:47,849 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 09:30:47,850 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-14 09:

FedPer: {'method': 'fedper', 'dataset': 'FD002', 'seed': 101, 'test_mae': 19.3652, 'nasa_score': 15701.43, 'comm_kb': 27650.0}


In [9]:
from run_experiment import run_simulation
print("Running FedPer (cluster-routed)...")
fedper_result = run_simulation('fedper', 'FD002', 202)
print("FedPer:", fedper_result)

Running FedPer (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8702, 30, 24), y shape = (8702,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (8994, 30, 24), y shape = (8994,)
✅ Created sequences: X shape = (2414, 30, 24), y shape = (2414,)
✅ Created sequences: X shape = (9462, 30, 24), y shape = (9462,)
✅ Created sequences: X shape = (2533, 30, 24), y shape = (2533,)
✅ Created sequences: X shape = (9248, 30, 24), y shape = (9248,)
✅ Created sequences: X shape = (2418, 30, 24), y shape = (2418,)


INFO flwr 2026-07-14 10:14:13,181 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 10:14:24,581	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 10:14:28,029 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'memory': 15107506176.0, 'CPU': 4.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6474645504.0}
INFO flwr 2026-07-14 10:14:28,030 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 10:14:28,060 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 10:14:28,061 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 10:14:28,061 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 10:14:28,062 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-14 10:

FedPer: {'method': 'fedper', 'dataset': 'FD002', 'seed': 202, 'test_mae': 19.3565, 'nasa_score': 5972.7, 'comm_kb': 27650.0}


In [10]:
from run_experiment import run_simulation
print("Running FedPer (cluster-routed)...")
fedper_result = run_simulation('fedper', 'FD002', 303)
print("FedPer:", fedper_result)

Running FedPer (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8924, 30, 24), y shape = (8924,)
✅ Created sequences: X shape = (2226, 30, 24), y shape = (2226,)
✅ Created sequences: X shape = (9116, 30, 24), y shape = (9116,)
✅ Created sequences: X shape = (2292, 30, 24), y shape = (2292,)
✅ Created sequences: X shape = (9556, 30, 24), y shape = (9556,)
✅ Created sequences: X shape = (2439, 30, 24), y shape = (2439,)
✅ Created sequences: X shape = (9280, 30, 24), y shape = (9280,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)


INFO flwr 2026-07-14 10:55:51,973 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 10:55:59,619	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 10:56:02,893 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'memory': 21359255143.0, 'GPU': 2.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 9153966489.0}
INFO flwr 2026-07-14 10:56:02,894 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 10:56:02,911 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 10:56:02,912 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 10:56:02,913 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 10:56:02,914 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-14 10:

FedPer: {'method': 'fedper', 'dataset': 'FD002', 'seed': 303, 'test_mae': 19.2621, 'nasa_score': 12899.13, 'comm_kb': 27650.0}


In [11]:
from run_experiment import run_simulation
print("Running FedPer (cluster-routed)...")
fedper_result = run_simulation('fedper', 'FD002', 2026)
print("FedPer:", fedper_result)

Running FedPer (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8764, 30, 24), y shape = (8764,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)
✅ Created sequences: X shape = (9013, 30, 24), y shape = (9013,)
✅ Created sequences: X shape = (2395, 30, 24), y shape = (2395,)
✅ Created sequences: X shape = (9164, 30, 24), y shape = (9164,)
✅ Created sequences: X shape = (2831, 30, 24), y shape = (2831,)
✅ Created sequences: X shape = (9304, 30, 24), y shape = (9304,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-14 11:36:45,311 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-14 11:36:53,717	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-14 11:36:57,011 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 9150205132.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'memory': 21350478644.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-14 11:36:57,012 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-14 11:36:57,031 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-14 11:36:57,032 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-14 11:36:57,034 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-14 11:36:57,035 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-14 11:

FedPer: {'method': 'fedper', 'dataset': 'FD002', 'seed': 2026, 'test_mae': 18.5416, 'nasa_score': 16486.82, 'comm_kb': 27650.0}


In [12]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD002', 42)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8805, 30, 24), y shape = (8805,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)
✅ Created sequences: X shape = (9138, 30, 24), y shape = (9138,)
✅ Created sequences: X shape = (2270, 30, 24), y shape = (2270,)
✅ Created sequences: X shape = (9413, 30, 24), y shape = (9413,)
✅ Created sequences: X shape = (2582, 30, 24), y shape = (2582,)
✅ Created sequences: X shape = (9283, 30, 24), y shape = (9283,)
✅ Created sequences: X shape = (2383, 30, 24), y shape = (2383,)
  [Round 1] Global MAE: 33.6682
  [Round 2] Global MAE: 22.8246
  [Round 3] Global MAE: 21.4481
  [Round 4] Global MAE: 20.9717
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [19.1, 19.1] | Avg: 19.1048
  [Round 6] Cluster MAEs: [19.82, 30.62] | Avg: 25.223
  [Round 7] Cluster MA

In [13]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD002', 101)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)
  [Round 1] Global MAE: 33.0877
  [Round 2] Global MAE: 17.6221
  [Round 3] Global MAE: 16.6045
  [Round 4] Global MAE: 16.4008
  [Round 5] CFL one-shot clustering: {0: 1, 1: 0, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [16.75, 16.75] | Avg: 16.7537
  [Round 6] Cluster MAEs: [17.28, 17.49] | Avg: 17.3877
  [Round 7] Cluster

In [14]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD002', 202)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8702, 30, 24), y shape = (8702,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (8994, 30, 24), y shape = (8994,)
✅ Created sequences: X shape = (2414, 30, 24), y shape = (2414,)
✅ Created sequences: X shape = (9462, 30, 24), y shape = (9462,)
✅ Created sequences: X shape = (2533, 30, 24), y shape = (2533,)
✅ Created sequences: X shape = (9248, 30, 24), y shape = (9248,)
✅ Created sequences: X shape = (2418, 30, 24), y shape = (2418,)
  [Round 1] Global MAE: 30.6837
  [Round 2] Global MAE: 17.9618
  [Round 3] Global MAE: 19.4105
  [Round 4] Global MAE: 18.0537
  [Round 5] CFL one-shot clustering: {0: 1, 1: 1, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [17.87, 17.87] | Avg: 17.8727
  [Round 6] Cluster MAEs: [19.95, 16.94] | Avg: 18.4451
  [Round 7] Cluster

In [5]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD002', 303)
print("CFL:", cfl_result)

E0000 00:00:1784096579.273030     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784096579.330272     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784096579.749252     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784096579.749290     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784096579.749293     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784096579.749295     113 computation_placer.cc:177] computation placer already registered. Please check linka

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8924, 30, 24), y shape = (8924,)
✅ Created sequences: X shape = (2226, 30, 24), y shape = (2226,)
✅ Created sequences: X shape = (9116, 30, 24), y shape = (9116,)
✅ Created sequences: X shape = (2292, 30, 24), y shape = (2292,)
✅ Created sequences: X shape = (9556, 30, 24), y shape = (9556,)
✅ Created sequences: X shape = (2439, 30, 24), y shape = (2439,)
✅ Created sequences: X shape = (9280, 30, 24), y shape = (9280,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)


I0000 00:00:1784096632.122158     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784096632.128034     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1784096638.466340     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Global MAE: 38.4017
  [Round 2] Global MAE: 16.6554
  [Round 3] Global MAE: 16.6577
  [Round 4] Global MAE: 15.4701
  [Round 5] CFL one-shot clustering: {0: 1, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [15.74, 15.74] | Avg: 15.7416
  [Round 6] Cluster MAEs: [19.83, 16.02] | Avg: 17.9214
  [Round 7] Cluster MAEs: [20.82, 15.87] | Avg: 18.3443
  [Round 8] Cluster MAEs: [19.69, 17.11] | Avg: 18.3993
  [Round 9] Cluster MAEs: [18.26, 16.55] | Avg: 17.4084
  [Round 10] Cluster MAEs: [22.6, 17.06] | Avg: 19.8312
  [Round 11] Cluster MAEs: [19.14, 16.99] | Avg: 18.0632
  [Round 12] Cluster MAEs: [21.38, 17.9] | Avg: 19.6381
  [Round 13] Cluster MAEs: [20.46, 18.53] | Avg: 19.4929
  [Round 14] Cluster MAEs: [21.86, 18.12] | Avg: 19.9938
  [Round 15] Cluster MAEs: [20.94, 18.76] | Avg: 19.8511
  [Round 16] Cluster MAEs: [21.4, 18.05] | Avg: 19.7268
  [Round 17] Cluster MAEs: [23.53, 18.23] | Avg: 20.8808
  [Round 18] Cluster MAEs: [22.56, 18.91] | Avg: 20.7326
  [Round 19] Cluster

In [6]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD002', 2026)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8764, 30, 24), y shape = (8764,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)
✅ Created sequences: X shape = (9013, 30, 24), y shape = (9013,)
✅ Created sequences: X shape = (2395, 30, 24), y shape = (2395,)
✅ Created sequences: X shape = (9164, 30, 24), y shape = (9164,)
✅ Created sequences: X shape = (2831, 30, 24), y shape = (2831,)
✅ Created sequences: X shape = (9304, 30, 24), y shape = (9304,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)
  [Round 1] Global MAE: 20.9381
  [Round 2] Global MAE: 16.4142
  [Round 3] Global MAE: 16.1653
  [Round 4] Global MAE: 15.074
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [15.14, 15.14] | Avg: 15.1392
  [Round 6] Cluster MAEs: [15.66, 15.73] | Avg: 15.6953
  [Round 7] Cluster 

In [7]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 42...")
result = run_acpfl('FD002', 42, alpha= 1.0)
print("AC-PFL 42:", result)

Checking AC-PFL seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8805, 30, 24), y shape = (8805,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)
✅ Created sequences: X shape = (9138, 30, 24), y shape = (9138,)
✅ Created sequences: X shape = (2270, 30, 24), y shape = (2270,)
✅ Created sequences: X shape = (9413, 30, 24), y shape = (9413,)
✅ Created sequences: X shape = (2582, 30, 24), y shape = (2582,)
✅ Created sequences: X shape = (9283, 30, 24), y shape = (9283,)
✅ Created sequences: X shape = (2383, 30, 24), y shape = (2383,)
  [Round 1] Val NASA per client: [40983.6, 48048.9, 68574.2, 35902.0]
  [Round 2] Val NASA per client: [23318.0, 56908.5, 64297.2, 27279.0]
  [Round 3] Val NASA per client: [18128.7, 34101.0, 43545.4, 25803.6]
  [Round 4] Val NASA per client: [25400.7, 28479.9, 47908.7, 31168.7]
  [Round 5] Re-clustered (α=1.0).

In [8]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 101...")
result = run_acpfl('FD002', 101, alpha= 1.0)
print("AC-PFL 101:", result)

Checking AC-PFL seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)
  [Round 1] Val NASA per client: [37648.3, 122723.9, 24887.5, 49341.5]
  [Round 2] Val NASA per client: [41297.6, 76337.3, 22367.6, 41221.5]
  [Round 3] Val NASA per client: [39825.6, 90645.6, 24121.5, 69363.9]
  [Round 4] Val NASA per client: [40771.0, 66229.5, 22149.4, 50173.0]
  [Round 5] Re-clustered (α=1.0

In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 202...")
result = run_acpfl('FD002', 202, alpha= 1.0)
print("AC-PFL 202:", result)

E0000 00:00:1784129269.654063     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784129269.708532     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784129270.126921     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784129270.126972     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784129270.126975     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784129270.126977     117 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8702, 30, 24), y shape = (8702,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (8994, 30, 24), y shape = (8994,)
✅ Created sequences: X shape = (2414, 30, 24), y shape = (2414,)
✅ Created sequences: X shape = (9462, 30, 24), y shape = (9462,)
✅ Created sequences: X shape = (2533, 30, 24), y shape = (2533,)
✅ Created sequences: X shape = (9248, 30, 24), y shape = (9248,)
✅ Created sequences: X shape = (2418, 30, 24), y shape = (2418,)


I0000 00:00:1784129328.611857     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784129328.618077     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1784129336.301525     166 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [38411.0, 159371.7, 237240.9, 18527.5]
  [Round 2] Val NASA per client: [37995.2, 104802.7, 30732.8, 20289.2]
  [Round 3] Val NASA per client: [43743.2, 113621.4, 29609.4, 16790.9]
  [Round 4] Val NASA per client: [40768.7, 137889.2, 33348.7, 19086.8]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 5] Cluster 0 val NASA: 62908.79
  [Round 5] Cluster 1 val NASA: 21922.24
  [Round 5] Val NASA per client: [35670.5, 90147.1, 25065.4, 18779.1]
  [Round 6] Val NASA per client: [44485.5, 98263.4, 24264.7, 22620.3]
  [Round 7] Val NASA per client: [47373.7, 85464.3, 24476.7, 24221.2]
  [Round 8] Val NASA per client: [52447.6, 117092.4, 22109.8, 19532.2]
  [Round 9] Val NASA per client: [46988.8, 92358.8, 25047.6, 17324.8]
  [Round 10] Re-clustered (α=1.0). Changes: {1: (0, 1), 2: (1, 0)}
  [Round 10] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 10] Cluster 0 val NASA: 39996.32
  [Round 1

In [7]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 303...")
result = run_acpfl('FD002', 303, alpha= 1.0)
print("AC-PFL 303:", result)

Checking AC-PFL seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8924, 30, 24), y shape = (8924,)
✅ Created sequences: X shape = (2226, 30, 24), y shape = (2226,)
✅ Created sequences: X shape = (9116, 30, 24), y shape = (9116,)
✅ Created sequences: X shape = (2292, 30, 24), y shape = (2292,)
✅ Created sequences: X shape = (9556, 30, 24), y shape = (9556,)
✅ Created sequences: X shape = (2439, 30, 24), y shape = (2439,)
✅ Created sequences: X shape = (9280, 30, 24), y shape = (9280,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)
  [Round 1] Val NASA per client: [26841.0, 57436.7, 28681.2, 28159.9]
  [Round 2] Val NASA per client: [27207.6, 62903.7, 21908.5, 46867.5]
  [Round 3] Val NASA per client: [26672.7, 45425.2, 22444.5, 27708.3]
  [Round 4] Val NASA per client: [18278.3, 35739.4, 21090.1, 47753.6]
  ⚠️ Spectral produced invalid c

In [8]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 2026...")
result = run_acpfl('FD002', 2026, alpha= 1.0)
print("AC-PFL 2026:", result)

Checking AC-PFL seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8764, 30, 24), y shape = (8764,)
✅ Created sequences: X shape = (2386, 30, 24), y shape = (2386,)
✅ Created sequences: X shape = (9013, 30, 24), y shape = (9013,)
✅ Created sequences: X shape = (2395, 30, 24), y shape = (2395,)
✅ Created sequences: X shape = (9164, 30, 24), y shape = (9164,)
✅ Created sequences: X shape = (2831, 30, 24), y shape = (2831,)
✅ Created sequences: X shape = (9304, 30, 24), y shape = (9304,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)
  [Round 1] Val NASA per client: [34127.8, 108663.9, 66386.0, 25922.2]
  [Round 2] Val NASA per client: [25020.4, 87663.4, 41803.0, 26034.4]
  [Round 3] Val NASA per client: [40065.5, 104529.8, 56367.4, 25876.2]
  [Round 4] Val NASA per client: [43846.0, 68905.0, 45595.5, 21002.1]
  [Round 5] Re-clustered (α=1

In [9]:
from run_experiment import run_simulation
print("Running FedProx...")
fedprox_result = run_simulation('fedprox', 'FD002', 42)
print("FedProx:", fedprox_result)

Running FedProx...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8805, 30, 24), y shape = (8805,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)
✅ Created sequences: X shape = (9138, 30, 24), y shape = (9138,)
✅ Created sequences: X shape = (2270, 30, 24), y shape = (2270,)
✅ Created sequences: X shape = (9413, 30, 24), y shape = (9413,)
✅ Created sequences: X shape = (2582, 30, 24), y shape = (2582,)
✅ Created sequences: X shape = (9283, 30, 24), y shape = (9283,)
✅ Created sequences: X shape = (2383, 30, 24), y shape = (2383,)


INFO flwr 2026-07-15 20:46:53,630 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-15 20:47:03,104	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-15 20:47:06,970 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'CPU': 4.0, 'memory': 12413777511.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 5320190361.0}
INFO flwr 2026-07-15 20:47:06,971 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-15 20:47:07,053 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-15 20:47:07,055 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-15 20:47:07,055 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-15 20:47:07,057 | server.py:91 | Evaluating initial parameters
(pid=96234) WARNING: All

  [Round 0] Test MAE: 74.8714 | NASA: 1540518.24


(DefaultActor pid=96236) I0000 00:00:1784148446.256750   96236 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=96237) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=96237) E0000 00:00:1784148428.454618   96237 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=96237) E0000 00:00:1784148428.487440   96237 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 38.9292 | NASA: 82747.82


...

  [Round 49] Test MAE: 19.8810 | NASA: 123809.88


DEBUG flwr 2026-07-15 21:46:25,642 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-15 21:46:25,643 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-15 21:47:24,160 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-15 21:47:37,516 | server.py:125 | fit progress: (50, 0.0, {'mae': 20.20121755563154, 'nasa_score': 90812.21983535563}, 3612.439043036)
DEBUG flwr 2026-07-15 21:47:37,517 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 20.2012 | NASA: 90812.22


DEBUG flwr 2026-07-15 21:47:41,150 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-15 21:47:41,152 | server.py:153 | FL finished in 3616.0749064660013
INFO flwr 2026-07-15 21:47:41,154 | app.py:225 | app_fit: losses_distributed [(1, 1886.997131653469), (2, 616.0592864576113), (3, 526.4821872878423), (4, 538.6844223946271), (5, 467.0315547967007), (6, 446.32901777613887), (7, 473.7492661987814), (8, 519.5155531562693), (9, 504.5629820152713), (10, 545.7516603919807), (11, 559.3191849105293), (12, 577.7678300256272), (13, 555.9633482548787), (14, 681.3422099836187), (15, 606.1499847730664), (16, 604.7999630476089), (17, 643.0654264191247), (18, 624.2663579124498), (19, 638.1797012870645), (20, 644.1264578227957), (21, 659.048521522092), (22, 635.6744772319754), (23, 735.2725140024078), (24, 726.5838600684307), (25, 656.7689660440656), (26, 650.4850574342095), (27, 666.2582245765001), (28, 667.0757811289491), (29, 645.9391584169392), (30, 636.187489

FedProx: {'method': 'fedprox', 'dataset': 'FD002', 'seed': 42, 'test_mae': 20.2012, 'nasa_score': 90812.22, 'comm_kb': 28900.78}


In [ ]:
from run_experiment import run_simulation
print("Running FedProx...")
fedprox_result = run_simulation('fedprox', 'FD002', 101)
print("FedProx:", fedprox_result)

Running FedProx...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)


INFO flwr 2026-07-15 21:48:51,039 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-15 21:49:06,436	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-15 21:49:10,580 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 2515295846.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0, 'memory': 5869023642.0}
INFO flwr 2026-07-15 21:49:10,582 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-15 21:49:10,613 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-15 21:49:10,617 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-15 21:49:10,621 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-15 21:49:10,623 | server.py:91 | Evaluating initial parameters
(pid=150106) WARNING: All

  [Round 0] Test MAE: 75.1588 | NASA: 1575361.75


(DefaultActor pid=150106) I0000 00:00:1784152171.356401  150106 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=150105) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=150105) E0000 00:00:1784152153.150842  150105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=150105) E0000 00:00:1784152153.180690  150105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=150105) W0000 00:00:1784152153.210890  150105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 41.0632 | NASA: 362115.32


...

DEBUG flwr 2026-07-15 22:03:20,556 | server.py:187 | evaluate_round 10 received 4 results and 0 failures
DEBUG flwr 2026-07-15 22:03:20,557 | server.py:222 | fit_round 11: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-15 22:04:27,643 | server.py:236 | fit_round 11 received 4 results and 0 failures
INFO flwr 2026-07-15 22:04:41,099 | server.py:125 | fit progress: (11, 0.0, {'mae': 16.27695474109134, 'nasa_score': 11965.775710117261}, 910.8095173139991)
DEBUG flwr 2026-07-15 22:04:41,101 | server.py:173 | evaluate_round 11: strategy sampled 4 clients (out of 4)


  [Round 11] Test MAE: 16.2770 | NASA: 11965.78


DEBUG flwr 2026-07-15 22:04:43,690 | server.py:187 | evaluate_round 11 received 4 results and 0 failures
DEBUG flwr 2026-07-15 22:04:43,691 | server.py:222 | fit_round 12: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-15 22:05:38,623 | server.py:236 | fit_round 12 received 4 results and 0 failures
INFO flwr 2026-07-15 22:05:51,951 | server.py:125 | fit progress: (12, 0.0, {'mae': 17.966096137941573, 'nasa_score': 11584.43540378572}, 981.6615014269992)
DEBUG flwr 2026-07-15 22:05:51,954 | server.py:173 | evaluate_round 12: strategy sampled 4 clients (out of 4)


  [Round 12] Test MAE: 17.9661 | NASA: 11584.44


DEBUG flwr 2026-07-15 22:05:55,137 | server.py:187 | evaluate_round 12 received 4 results and 0 failures
DEBUG flwr 2026-07-15 22:05:55,138 | server.py:222 | fit_round 13: strategy sampled 4 clients (out of 4)


In [8]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD001', 202)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3228, 30, 24), y shape = (3228,)
✅ Created sequences: X shape = (978, 30, 24), y shape = (978,)
✅ Created sequences: X shape = (3704, 30, 24), y shape = (3704,)
✅ Created sequences: X shape = (1250, 30, 24), y shape = (1250,)
✅ Created sequences: X shape = (3407, 30, 24), y shape = (3407,)
✅ Created sequences: X shape = (807, 30, 24), y shape = (807,)
✅ Created sequences: X shape = (3378, 30, 24), y shape = (3378,)
✅ Created sequences: X shape = (979, 30, 24), y shape = (979,)


INFO flwr 2026-07-18 07:52:38,461 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
Exception in thread Thread-359:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1433, in run
    self.function(*self.args, **self.kwargs)
  File "/usr/local/lib/python3.12/dist-packages/flwr/simulation/app.py", line 258, in update_resources
    num_max_actors = pool_size_from_resources(client_resources)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flwr/simulation/ray_transport/ray_actor.py", line 107, in pool_size_from_resources
    nodes = ray.nodes()
            ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/client_mode_hook.py", line 107, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/py

Ditto: {'method': 'ditto', 'dataset': 'FD001', 'seed': 202, 'test_mae': 13.1749, 'nasa_score': 444.12, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD001', 303)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3238, 30, 24), y shape = (3238,)
✅ Created sequences: X shape = (968, 30, 24), y shape = (968,)
✅ Created sequences: X shape = (3950, 30, 24), y shape = (3950,)
✅ Created sequences: X shape = (1004, 30, 24), y shape = (1004,)
✅ Created sequences: X shape = (3321, 30, 24), y shape = (3321,)
✅ Created sequences: X shape = (893, 30, 24), y shape = (893,)
✅ Created sequences: X shape = (3460, 30, 24), y shape = (3460,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)


INFO flwr 2026-07-18 08:36:19,990 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 08:36:28,291	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 08:36:31,706 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'GPU': 2.0, 'memory': 21497236276.0, 'object_store_memory': 9213101260.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-18 08:36:31,707 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 08:36:31,726 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 08:36:31,727 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 08:36:31,728 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 08:36:31,730 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-18 08:

Ditto: {'method': 'ditto', 'dataset': 'FD001', 'seed': 303, 'test_mae': 14.5076, 'nasa_score': 471.08, 'comm_kb': 28900.78}


In [10]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD001', 2026)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3350, 30, 24), y shape = (3350,)
✅ Created sequences: X shape = (856, 30, 24), y shape = (856,)
✅ Created sequences: X shape = (3729, 30, 24), y shape = (3729,)
✅ Created sequences: X shape = (1225, 30, 24), y shape = (1225,)
✅ Created sequences: X shape = (3346, 30, 24), y shape = (3346,)
✅ Created sequences: X shape = (868, 30, 24), y shape = (868,)
✅ Created sequences: X shape = (3312, 30, 24), y shape = (3312,)
✅ Created sequences: X shape = (1045, 30, 24), y shape = (1045,)


INFO flwr 2026-07-18 09:21:06,767 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 09:21:15,575	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 09:21:19,414 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 8183710924.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'memory': 19095325492.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0}
INFO flwr 2026-07-18 09:21:19,414 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 09:21:19,432 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 09:21:19,433 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 09:21:19,435 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 09:21:19,435 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-18 09:

Ditto: {'method': 'ditto', 'dataset': 'FD001', 'seed': 2026, 'test_mae': 14.2638, 'nasa_score': 447.31, 'comm_kb': 28900.78}


In [11]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD001', 42)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3281, 30, 24), y shape = (3281,)
✅ Created sequences: X shape = (925, 30, 24), y shape = (925,)
✅ Created sequences: X shape = (3803, 30, 24), y shape = (3803,)
✅ Created sequences: X shape = (1151, 30, 24), y shape = (1151,)
✅ Created sequences: X shape = (3270, 30, 24), y shape = (3270,)
✅ Created sequences: X shape = (944, 30, 24), y shape = (944,)
✅ Created sequences: X shape = (3582, 30, 24), y shape = (3582,)
✅ Created sequences: X shape = (775, 30, 24), y shape = (775,)


INFO flwr 2026-07-18 10:05:55,627 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 10:06:03,948	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 10:06:07,315 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 21491731252.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 9210741964.0, 'CPU': 4.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-18 10:06:07,316 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 10:06:07,334 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 10:06:07,335 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 10:06:07,336 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 10:06:07,337 | server.py:91 | Evaluating initial parameters
(pid=359785) WARNING: Al

  [Round 0] Test MAE: 75.4688 | NASA: 422798.67


(DefaultActor pid=359785) I0000 00:00:1784369176.940518  359785 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=359786) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=359786) E0000 00:00:1784369168.653335  359786 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=359786) E0000 00:00:1784369168.671405  359786 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=359786) W0000 00:00:1784369168.711929  359786 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 25.6006 | NASA: 2927.80


...

  [Round 49] Test MAE: 11.7509 | NASA: 390.51


DEBUG flwr 2026-07-18 10:30:39,526 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-18 10:30:39,527 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-18 10:31:04,385 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 10:31:05,718 | server.py:125 | fit progress: (50, 0.0, {'mae': 12.320286445617675, 'nasa_score': 441.94307288900467}, 1496.2506755890008)
DEBUG flwr 2026-07-18 10:31:05,719 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 12.3203 | NASA: 441.94


DEBUG flwr 2026-07-18 10:31:07,800 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 10:31:07,800 | server.py:153 | FL finished in 1498.332777048001
INFO flwr 2026-07-18 10:31:07,801 | app.py:225 | app_fit: losses_distributed [(1, 1201.3603467536695), (2, 421.5342115158463), (3, 373.14360989657314), (4, 366.6438378335302), (5, 395.3011797637336), (6, 338.45212973291854), (7, 386.2113879901147), (8, 370.11008499808935), (9, 361.12380914700674), (10, 341.7036240970979), (11, 334.6253017380304), (12, 332.1371520031111), (13, 396.3206711559271), (14, 347.1086024629897), (15, 344.5008471100698), (16, 321.6164764766165), (17, 300.6135754088953), (18, 324.18285501150905), (19, 386.82317345660664), (20, 356.7893616808262), (21, 336.6483954830446), (22, 359.49604534405495), (23, 310.6225053387668), (24, 299.3370742616917), (25, 301.6284651885705), (26, 313.9932296974072), (27, 308.95504984296514), (28, 312.1279246132993), (29, 324.8419170555547), (30, 33

FedProx: {'method': 'fedprox', 'dataset': 'FD001', 'seed': 42, 'test_mae': 12.3203, 'nasa_score': 441.94, 'comm_kb': 28900.78}


In [12]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD001', 101)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3249, 30, 24), y shape = (3249,)
✅ Created sequences: X shape = (957, 30, 24), y shape = (957,)
✅ Created sequences: X shape = (3588, 30, 24), y shape = (3588,)
✅ Created sequences: X shape = (1366, 30, 24), y shape = (1366,)
✅ Created sequences: X shape = (3317, 30, 24), y shape = (3317,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)
✅ Created sequences: X shape = (3575, 30, 24), y shape = (3575,)
✅ Created sequences: X shape = (782, 30, 24), y shape = (782,)


INFO flwr 2026-07-18 10:31:28,989 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 10:31:37,540	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 10:31:40,934 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'memory': 21489133568.0, 'CPU': 4.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0, 'object_store_memory': 9209628672.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-18 10:31:40,934 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 10:31:40,953 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 10:31:40,954 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 10:31:40,955 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 10:31:40,955 | server.py:91 | Evaluating initial parameters
(pid=413288) WARNING: Al

  [Round 0] Test MAE: 74.2934 | NASA: 384536.81


(DefaultActor pid=413289) I0000 00:00:1784370710.941555  413289 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=413287) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=413287) E0000 00:00:1784370702.184422  413287 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=413287) E0000 00:00:1784370702.198201  413287 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=413287) W0000 00:00:1784370702.236412  413287 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 25.3884 | NASA: 4666.51


...

  [Round 49] Test MAE: 10.2629 | NASA: 257.43


DEBUG flwr 2026-07-18 10:56:08,321 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-18 10:56:08,321 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-18 10:56:31,025 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 10:56:32,348 | server.py:125 | fit progress: (50, 0.0, {'mae': 10.47062349319458, 'nasa_score': 273.17079048305345}, 1489.254663687001)
DEBUG flwr 2026-07-18 10:56:32,349 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 10.4706 | NASA: 273.17


DEBUG flwr 2026-07-18 10:56:35,714 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 10:56:35,716 | server.py:153 | FL finished in 1492.6222011770005
INFO flwr 2026-07-18 10:56:35,717 | app.py:225 | app_fit: losses_distributed [(1, 1527.1013855407502), (2, 687.3309536714187), (3, 521.5711955576166), (4, 569.1791796820096), (5, 453.6002498704871), (6, 513.6326877645466), (7, 590.8251549121679), (8, 555.6571815601294), (9, 477.74717815728974), (10, 435.4592054177379), (11, 440.2291824516209), (12, 350.89608970920426), (13, 417.15759230065146), (14, 433.41389377256564), (15, 420.1602132474107), (16, 415.4807344328934), (17, 388.54543660176745), (18, 437.2198503447556), (19, 468.9082673514622), (20, 339.51712903542733), (21, 366.9097555485563), (22, 459.2409799548163), (23, 366.07102790157654), (24, 407.0732598635985), (25, 385.04829719387135), (26, 365.3032696832126), (27, 374.30538134405697), (28, 370.9019184903703), (29, 370.4917729123719), (30, 

FedProx: {'method': 'fedprox', 'dataset': 'FD001', 'seed': 101, 'test_mae': 10.4706, 'nasa_score': 273.17, 'comm_kb': 28900.78}


In [13]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD001', 202)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3228, 30, 24), y shape = (3228,)
✅ Created sequences: X shape = (978, 30, 24), y shape = (978,)
✅ Created sequences: X shape = (3704, 30, 24), y shape = (3704,)
✅ Created sequences: X shape = (1250, 30, 24), y shape = (1250,)
✅ Created sequences: X shape = (3407, 30, 24), y shape = (3407,)
✅ Created sequences: X shape = (807, 30, 24), y shape = (807,)
✅ Created sequences: X shape = (3378, 30, 24), y shape = (3378,)
✅ Created sequences: X shape = (979, 30, 24), y shape = (979,)


INFO flwr 2026-07-18 10:56:52,580 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 10:57:04,278	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 10:57:07,911 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'memory': 15076666573.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 6461428531.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-18 10:57:07,912 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 10:57:07,938 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 10:57:07,939 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 10:57:07,940 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 10:57:07,941 | server.py:91 | Evaluating initial parameters
(pid=466680) WARNING: Al

  [Round 0] Test MAE: 73.2113 | NASA: 354530.78


(DefaultActor pid=466680) I0000 00:00:1784372240.543590  466680 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=466679) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=466679) E0000 00:00:1784372230.880807  466679 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=466679) E0000 00:00:1784372230.911596  466679 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=466679) W0000 00:00:1784372230.972718  466679 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 21.9856 | NASA: 2155.49


...

  [Round 49] Test MAE: 11.6731 | NASA: 408.01


DEBUG flwr 2026-07-18 11:20:50,027 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-18 11:20:50,027 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-18 11:21:13,221 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 11:21:14,577 | server.py:125 | fit progress: (50, 0.0, {'mae': 11.821823444366455, 'nasa_score': 440.1844084043561}, 1444.5096407699984)
DEBUG flwr 2026-07-18 11:21:14,578 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 11.8218 | NASA: 440.18


DEBUG flwr 2026-07-18 11:21:16,695 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 11:21:16,696 | server.py:153 | FL finished in 1446.6288618009985
INFO flwr 2026-07-18 11:21:16,697 | app.py:225 | app_fit: losses_distributed [(1, 1264.5148394346593), (2, 568.1212240883409), (3, 472.775265023194), (4, 553.4722776199136), (5, 544.5916439753952), (6, 468.68970833147824), (7, 519.8373079513755), (8, 445.98584413552203), (9, 436.1019958093146), (10, 490.9293478075758), (11, 454.13137932565), (12, 372.97203888486854), (13, 430.6744379785801), (14, 382.2161674176393), (15, 439.7131850732519), (16, 376.0968311501785), (17, 422.70397597209814), (18, 388.3443382768769), (19, 430.8853358756742), (20, 425.4113709887403), (21, 459.1175898697819), (22, 417.2130983254775), (23, 421.3916038205271), (24, 427.30033631982883), (25, 347.15287529869823), (26, 396.962300196535), (27, 421.2086495787692), (28, 354.69573144005136), (29, 370.82039927058275), (30, 350.1

FedProx: {'method': 'fedprox', 'dataset': 'FD001', 'seed': 202, 'test_mae': 11.8218, 'nasa_score': 440.18, 'comm_kb': 28900.78}


In [14]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD001', 303)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3238, 30, 24), y shape = (3238,)
✅ Created sequences: X shape = (968, 30, 24), y shape = (968,)
✅ Created sequences: X shape = (3950, 30, 24), y shape = (3950,)
✅ Created sequences: X shape = (1004, 30, 24), y shape = (1004,)
✅ Created sequences: X shape = (3321, 30, 24), y shape = (3321,)
✅ Created sequences: X shape = (893, 30, 24), y shape = (893,)
✅ Created sequences: X shape = (3460, 30, 24), y shape = (3460,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)


INFO flwr 2026-07-18 11:21:33,969 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 11:21:46,288	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 11:21:50,102 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15115173069.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 6477931315.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'GPU': 2.0}
INFO flwr 2026-07-18 11:21:50,103 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 11:21:50,137 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 11:21:50,138 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 11:21:50,140 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 11:21:50,140 | server.py:91 | Evaluating initial parameters
(pid=519044) WARNING: Al

  [Round 0] Test MAE: 74.9885 | NASA: 403603.74


(DefaultActor pid=519044) I0000 00:00:1784373722.879707  519044 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=519046) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=519046) E0000 00:00:1784373713.009419  519046 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=519046) E0000 00:00:1784373713.037782  519046 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=519046) W0000 00:00:1784373713.099366  519046 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 26.3637 | NASA: 5504.37


...

  [Round 49] Test MAE: 12.5901 | NASA: 509.36


DEBUG flwr 2026-07-18 11:46:27,576 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-18 11:46:27,578 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-18 11:46:49,823 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 11:46:51,182 | server.py:125 | fit progress: (50, 0.0, {'mae': 11.7327570438385, 'nasa_score': 389.2916201852627}, 1498.9028082490004)
DEBUG flwr 2026-07-18 11:46:51,184 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 11.7328 | NASA: 389.29


DEBUG flwr 2026-07-18 11:46:54,553 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 11:46:54,555 | server.py:153 | FL finished in 1502.275112104002
INFO flwr 2026-07-18 11:46:54,556 | app.py:225 | app_fit: losses_distributed [(1, 1201.1525990690468), (2, 350.12240632628584), (3, 339.3434620136786), (4, 342.3033182934077), (5, 354.06201919807137), (6, 337.2273904981922), (7, 376.10050539283407), (8, 336.1302509622204), (9, 368.23112686944097), (10, 305.5799300697241), (11, 347.1514845447043), (12, 337.14006619653696), (13, 340.8980885515309), (14, 336.74311975441606), (15, 394.55714386286473), (16, 297.2220106555831), (17, 324.21827448713594), (18, 327.98453198260034), (19, 339.31009941567515), (20, 345.85922929316615), (21, 367.8186898583874), (22, 330.1139415272495), (23, 325.16306540181444), (24, 364.02946941045167), (25, 336.1890270551553), (26, 340.12690137004296), (27, 339.34015616985283), (28, 301.36420697420596), (29, 299.92209001426556)

FedProx: {'method': 'fedprox', 'dataset': 'FD001', 'seed': 303, 'test_mae': 11.7328, 'nasa_score': 389.29, 'comm_kb': 28900.78}


In [15]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD001', 2026)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3350, 30, 24), y shape = (3350,)
✅ Created sequences: X shape = (856, 30, 24), y shape = (856,)
✅ Created sequences: X shape = (3729, 30, 24), y shape = (3729,)
✅ Created sequences: X shape = (1225, 30, 24), y shape = (1225,)
✅ Created sequences: X shape = (3346, 30, 24), y shape = (3346,)
✅ Created sequences: X shape = (868, 30, 24), y shape = (868,)
✅ Created sequences: X shape = (3312, 30, 24), y shape = (3312,)
✅ Created sequences: X shape = (1045, 30, 24), y shape = (1045,)


INFO flwr 2026-07-18 11:47:11,717 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 11:47:23,450	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 11:47:27,082 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 6466949529.0, 'memory': 15089548903.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0, 'CPU': 4.0}
INFO flwr 2026-07-18 11:47:27,085 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 11:47:27,113 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 11:47:27,114 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 11:47:27,116 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 11:47:27,119 | server.py:91 | Evaluating initial parameters
(pid=572289) WARNING: Al

  [Round 0] Test MAE: 74.2340 | NASA: 383602.12


(DefaultActor pid=572289) I0000 00:00:1784375260.173594  572289 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=572288) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=572288) E0000 00:00:1784375249.426350  572288 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=572288) E0000 00:00:1784375249.441059  572288 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=572288) W0000 00:00:1784375249.476246  572288 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 26.1477 | NASA: 6805.10


...

  [Round 49] Test MAE: 10.6070 | NASA: 319.23


DEBUG flwr 2026-07-18 12:13:28,576 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-18 12:13:28,577 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-18 12:13:57,227 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 12:13:58,776 | server.py:125 | fit progress: (50, 0.0, {'mae': 11.100579204559326, 'nasa_score': 314.04592078800596}, 1588.2191398299983)
DEBUG flwr 2026-07-18 12:13:58,777 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 11.1006 | NASA: 314.05


DEBUG flwr 2026-07-18 12:14:02,673 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-18 12:14:02,674 | server.py:153 | FL finished in 1592.1173438509977
INFO flwr 2026-07-18 12:14:02,675 | app.py:225 | app_fit: losses_distributed [(1, 933.8912575711712), (2, 426.7129100098512), (3, 445.7414086446442), (4, 398.8097242222587), (5, 391.86419609730757), (6, 357.9290899927161), (7, 364.2188469089743), (8, 310.3848359896412), (9, 356.493570311522), (10, 370.95182638306824), (11, 327.1245619726587), (12, 318.7580633492964), (13, 316.87042186662563), (14, 356.26971693807803), (15, 303.7999076098278), (16, 321.7166295173351), (17, 313.4321232443758), (18, 328.003122580905), (19, 312.43929566988425), (20, 323.69247651995573), (21, 380.3086470526102), (22, 323.86220394281605), (23, 311.32041121985714), (24, 348.49021141422827), (25, 286.0881875410161), (26, 297.87395453226225), (27, 302.77391739923115), (28, 299.2235139437061), (29, 302.54434459306145), (30, 

FedProx: {'method': 'fedprox', 'dataset': 'FD001', 'seed': 2026, 'test_mae': 11.1006, 'nasa_score': 314.05, 'comm_kb': 28900.78}


In [16]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD001', 101)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3249, 30, 24), y shape = (3249,)
✅ Created sequences: X shape = (957, 30, 24), y shape = (957,)
✅ Created sequences: X shape = (3588, 30, 24), y shape = (3588,)
✅ Created sequences: X shape = (1366, 30, 24), y shape = (1366,)
✅ Created sequences: X shape = (3317, 30, 24), y shape = (3317,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)
✅ Created sequences: X shape = (3575, 30, 24), y shape = (3575,)
✅ Created sequences: X shape = (782, 30, 24), y shape = (782,)


INFO flwr 2026-07-18 12:14:22,073 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-18 12:14:35,100	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-18 12:14:39,127 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6469708185.0, 'memory': 15095985767.0, 'CPU': 4.0, 'GPU': 2.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-18 12:14:39,128 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-18 12:14:39,159 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-18 12:14:39,160 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-18 12:14:39,161 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-18 12:14:39,162 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-18 12:

Ditto: {'method': 'ditto', 'dataset': 'FD001', 'seed': 101, 'test_mae': 14.0041, 'nasa_score': 526.7, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 404...")
result = run_acpfl('FD002', 404, alpha= 1.0)
print("AC-PFL 404:", result)

Checking AC-PFL seed 404...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8825, 30, 24), y shape = (8825,)
✅ Created sequences: X shape = (2325, 30, 24), y shape = (2325,)
✅ Created sequences: X shape = (9274, 30, 24), y shape = (9274,)
✅ Created sequences: X shape = (2134, 30, 24), y shape = (2134,)
✅ Created sequences: X shape = (9407, 30, 24), y shape = (9407,)
✅ Created sequences: X shape = (2588, 30, 24), y shape = (2588,)
✅ Created sequences: X shape = (9394, 30, 24), y shape = (9394,)
✅ Created sequences: X shape = (2272, 30, 24), y shape = (2272,)
  [Round 1] Val NASA per client: [52301.6, 44132.7, 34153.2, 78171.6]
  [Round 2] Val NASA per client: [29840.5, 37256.5, 36762.0, 79288.8]
  [Round 3] Val NASA per client: [26085.9, 42723.1, 27724.3, 73717.6]
  [Round 4] Val NASA per client: [27661.7, 34824.8, 30700.7, 90171.8]
  ⚠️ Spectral produced invalid c

In [ ]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 505...")
result = run_acpfl('FD002', 505, alpha= 1.0)
print("AC-PFL 505:", result)

Checking AC-PFL seed 505...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8965, 30, 24), y shape = (8965,)
✅ Created sequences: X shape = (2185, 30, 24), y shape = (2185,)
✅ Created sequences: X shape = (9074, 30, 24), y shape = (9074,)
✅ Created sequences: X shape = (2334, 30, 24), y shape = (2334,)
✅ Created sequences: X shape = (9249, 30, 24), y shape = (9249,)
✅ Created sequences: X shape = (2746, 30, 24), y shape = (2746,)
✅ Created sequences: X shape = (9412, 30, 24), y shape = (9412,)
✅ Created sequences: X shape = (2254, 30, 24), y shape = (2254,)
  [Round 1] Val NASA per client: [23511.4, 20840.7, 49316.3, 48255.6]
  [Round 2] Val NASA per client: [32793.3, 22376.2, 32862.3, 24461.7]
  [Round 3] Val NASA per client: [19785.7, 25666.9, 59406.2, 61356.8]
  [Round 4] Val NASA per client: [31745.3, 21060.5, 52770.5, 24022.9]
  [Round 5] Re-clustered (α=1.0)

In [ ]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 606...")
result = run_acpfl('FD002', 606, alpha= 1.0)
print("AC-PFL 606:", result)

Checking AC-PFL seed 606...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (9058, 30, 24), y shape = (9058,)
✅ Created sequences: X shape = (2092, 30, 24), y shape = (2092,)
✅ Created sequences: X shape = (9284, 30, 24), y shape = (9284,)
✅ Created sequences: X shape = (2124, 30, 24), y shape = (2124,)
✅ Created sequences: X shape = (9314, 30, 24), y shape = (9314,)
✅ Created sequences: X shape = (2681, 30, 24), y shape = (2681,)
✅ Created sequences: X shape = (9443, 30, 24), y shape = (9443,)
✅ Created sequences: X shape = (2223, 30, 24), y shape = (2223,)
  [Round 1] Val NASA per client: [106132.7, 33905.4, 60618.7, 47020.3]
  [Round 2] Val NASA per client: [56353.8, 50466.6, 90555.9, 65033.8]
  [Round 3] Val NASA per client: [58590.7, 39929.0, 53753.3, 43865.4]
  [Round 4] Val NASA per client: [51420.2, 36263.5, 53254.8, 61708.1]
  [Round 5] Re-clustered (α=1.0

In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 707...")
result = run_acpfl('FD002', 707, alpha= 1.0)
print("AC-PFL 707:", result)

E0000 00:00:1784471573.649638     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784471573.720718     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784471574.307720     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784471574.307769     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784471574.307771     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784471574.307774     117 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 707...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8855, 30, 24), y shape = (8855,)
✅ Created sequences: X shape = (2295, 30, 24), y shape = (2295,)
✅ Created sequences: X shape = (9348, 30, 24), y shape = (9348,)
✅ Created sequences: X shape = (2060, 30, 24), y shape = (2060,)
✅ Created sequences: X shape = (9715, 30, 24), y shape = (9715,)
✅ Created sequences: X shape = (2280, 30, 24), y shape = (2280,)
✅ Created sequences: X shape = (9124, 30, 24), y shape = (9124,)
✅ Created sequences: X shape = (2542, 30, 24), y shape = (2542,)


I0000 00:00:1784471629.625054     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784471629.630836     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1784471636.828957     167 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [34972.0, 118832.4, 56263.0, 234958.4]
  [Round 2] Val NASA per client: [63353.2, 60631.5, 53262.1, 60843.9]
  [Round 3] Val NASA per client: [55423.8, 37432.0, 40534.1, 109912.8]
  [Round 4] Val NASA per client: [29540.5, 35557.3, 43895.3, 46226.5]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 5] Cluster 0 val NASA: 40806.43
  [Round 5] Cluster 1 val NASA: 38623.88
  [Round 5] Val NASA per client: [46938.7, 34674.2, 41298.5, 35949.3]
  [Round 6] Val NASA per client: [59987.2, 38520.6, 49598.8, 45863.3]
  [Round 7] Val NASA per client: [45294.5, 35534.2, 75911.4, 32345.5]
  [Round 8] Val NASA per client: [53286.1, 33330.7, 51186.5, 43996.7]
  [Round 9] Val NASA per client: [78827.2, 53311.7, 69774.7, 43541.3]
  [Round 10] Re-clustered (α=1.0). Changes: {1: (0, 1), 3: (1, 0)}
  [Round 10] Assignments: {'0': 0, '1': 1, '2': 1, '3': 0}
  [Round 10] Cluster 0 val NASA: 41535.62
  [Round 10] 

In [7]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 808...")
result = run_acpfl('FD002', 808, alpha= 1.0)
print("AC-PFL 808:", result)

Checking AC-PFL seed 808...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (9062, 30, 24), y shape = (9062,)
✅ Created sequences: X shape = (2088, 30, 24), y shape = (2088,)
✅ Created sequences: X shape = (8975, 30, 24), y shape = (8975,)
✅ Created sequences: X shape = (2433, 30, 24), y shape = (2433,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2249, 30, 24), y shape = (2249,)
✅ Created sequences: X shape = (9426, 30, 24), y shape = (9426,)
✅ Created sequences: X shape = (2240, 30, 24), y shape = (2240,)
  [Round 1] Val NASA per client: [55264.4, 29822.9, 76969.3, 69416.6]
  [Round 2] Val NASA per client: [34218.2, 32544.6, 124880.1, 36181.8]
  [Round 3] Val NASA per client: [33984.9, 29823.7, 61878.0, 29335.2]
  [Round 4] Val NASA per client: [34033.3, 41625.3, 82831.2, 40421.0]
  ⚠️ Spectral produced invalid 

In [8]:
from run_experiment import run_simulation
print("Checking FedAvg seed 404...")
result = run_simulation('fedavg', 'FD002', 404)
print("FedAvg 404:", result)

Checking FedAvg seed 404...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8825, 30, 24), y shape = (8825,)
✅ Created sequences: X shape = (2325, 30, 24), y shape = (2325,)
✅ Created sequences: X shape = (9274, 30, 24), y shape = (9274,)
✅ Created sequences: X shape = (2134, 30, 24), y shape = (2134,)
✅ Created sequences: X shape = (9407, 30, 24), y shape = (9407,)
✅ Created sequences: X shape = (2588, 30, 24), y shape = (2588,)
✅ Created sequences: X shape = (9394, 30, 24), y shape = (9394,)
✅ Created sequences: X shape = (2272, 30, 24), y shape = (2272,)


INFO flwr 2026-07-19 18:35:13,461 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-19 18:35:22,092	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-19 18:35:25,619 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15613698868.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6691585228.0}
INFO flwr 2026-07-19 18:35:25,620 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-19 18:35:25,706 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-19 18:35:25,708 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-19 18:35:25,709 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-19 18:35:25,709 | server.py:91 | Evaluating initial parameters
(pid=62003) WARNING: All

  [Round 0] Test MAE: 73.7648 | NASA: 1411410.08


(DefaultActor pid=62003) I0000 00:00:1784486138.965254   62003 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=62001) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=62001) E0000 00:00:1784486126.814136   62001 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=62004) E0000 00:00:1784486126.897713   62004 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 36.8026 | NASA: 69618.90


...

  [Round 49] Test MAE: 18.7403 | NASA: 7978.80


DEBUG flwr 2026-07-19 19:22:24,341 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-19 19:22:24,342 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-19 19:23:12,005 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 19:23:20,790 | server.py:125 | fit progress: (50, 0.0, {'mae': 19.3770795623308, 'nasa_score': 9127.786267509524}, 2862.865702787998)
DEBUG flwr 2026-07-19 19:23:20,791 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 19.3771 | NASA: 9127.79


DEBUG flwr 2026-07-19 19:23:22,950 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 19:23:22,950 | server.py:153 | FL finished in 2865.0260704679986
INFO flwr 2026-07-19 19:23:22,952 | app.py:225 | app_fit: losses_distributed [(1, 1640.9087536268728), (2, 755.3161740164373), (3, 632.8463654930032), (4, 506.9489270760939), (5, 547.6875716879819), (6, 449.626701436031), (7, 459.10090254222723), (8, 446.16082406067034), (9, 487.11803101843446), (10, 533.5632848542115), (11, 528.0640469291254), (12, 561.0396238053379), (13, 581.4407327707034), (14, 526.3495093995582), (15, 595.2141924867406), (16, 604.6610638360274), (17, 620.1411402588021), (18, 624.9757463881125), (19, 629.5974984781491), (20, 631.0689729161882), (21, 618.3231254747107), (22, 606.913264576183), (23, 634.0511344797507), (24, 639.9588197392319), (25, 676.339327436722), (26, 665.2173268309682), (27, 646.1121465161272), (28, 640.2668601645076), (29, 680.1345353890468), (30, 642.88337

FedAvg 404: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 404, 'test_mae': 19.3771, 'nasa_score': 9127.79, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Checking FedAvg seed 505...")
result = run_simulation('fedavg', 'FD002', 505)
print("FedAvg 505:", result)

Checking FedAvg seed 505...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8965, 30, 24), y shape = (8965,)
✅ Created sequences: X shape = (2185, 30, 24), y shape = (2185,)
✅ Created sequences: X shape = (9074, 30, 24), y shape = (9074,)
✅ Created sequences: X shape = (2334, 30, 24), y shape = (2334,)
✅ Created sequences: X shape = (9249, 30, 24), y shape = (9249,)
✅ Created sequences: X shape = (2746, 30, 24), y shape = (2746,)
✅ Created sequences: X shape = (9412, 30, 24), y shape = (9412,)
✅ Created sequences: X shape = (2254, 30, 24), y shape = (2254,)


INFO flwr 2026-07-19 19:24:17,605 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-19 19:24:30,222	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-19 19:24:33,686 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'memory': 9385277440.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 4022261760.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-19 19:24:33,687 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-19 19:24:33,718 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-19 19:24:33,719 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-19 19:24:33,720 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-19 19:24:33,722 | server.py:91 | Evaluating initial parameters
(pid=115937) WARNING: All

  [Round 0] Test MAE: 74.5552 | NASA: 1505824.32


(DefaultActor pid=115937) I0000 00:00:1784489087.616702  115937 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(DefaultActor pid=115937) I0000 00:00:1784489097.698580  116304 cuda_dnn.cc:529] Loaded cuDNN version 91002
(pid=115936) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=115936) E0000 00:00:1784489076.378019  115936 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=115936) E0000 00:00:1784489076.423454  115936 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=115936) W0000 00:00:1784489076.527571  115936 computation_placer.cc:177] c

  [Round 1] Test MAE: 35.6277 | NASA: 63841.70


...

  [Round 49] Test MAE: 20.1869 | NASA: 16776.52


DEBUG flwr 2026-07-19 20:11:11,847 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-19 20:11:11,849 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-19 20:12:04,132 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 20:12:12,977 | server.py:125 | fit progress: (50, 0.0, {'mae': 20.194704799578457, 'nasa_score': 15913.330690885705}, 2846.702146964999)
DEBUG flwr 2026-07-19 20:12:12,978 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 20.1947 | NASA: 15913.33


DEBUG flwr 2026-07-19 20:12:16,606 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 20:12:16,608 | server.py:153 | FL finished in 2850.3328733300004
INFO flwr 2026-07-19 20:12:16,609 | app.py:225 | app_fit: losses_distributed [(1, 1554.3736048923945), (2, 601.7271801226905), (3, 625.3013422457357), (4, 515.5861905980904), (5, 533.337518547147), (6, 579.7836207083161), (7, 561.8610676083073), (8, 631.1056531827381), (9, 584.0149091657864), (10, 612.9734508675905), (11, 629.7817146276424), (12, 613.2165442578043), (13, 636.5834381170841), (14, 652.323490688237), (15, 663.1620611893503), (16, 634.4423070427299), (17, 633.1223291556792), (18, 648.8903337829892), (19, 678.8864222495592), (20, 702.8730964648622), (21, 692.1436074320314), (22, 671.8162670277762), (23, 718.1572479014011), (24, 674.7898860353922), (25, 670.5592693621802), (26, 667.4080127035197), (27, 674.6951331647458), (28, 695.2262624024669), (29, 708.4643091105003), (30, 672.6563225

FedAvg 505: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 505, 'test_mae': 20.1947, 'nasa_score': 15913.33, 'comm_kb': 28900.78}


In [10]:
from run_experiment import run_simulation
print("Checking FedAvg seed 606...")
result = run_simulation('fedavg', 'FD002', 606)
print("FedAvg 606:", result)

Checking FedAvg seed 606...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (9058, 30, 24), y shape = (9058,)
✅ Created sequences: X shape = (2092, 30, 24), y shape = (2092,)
✅ Created sequences: X shape = (9284, 30, 24), y shape = (9284,)
✅ Created sequences: X shape = (2124, 30, 24), y shape = (2124,)
✅ Created sequences: X shape = (9314, 30, 24), y shape = (9314,)
✅ Created sequences: X shape = (2681, 30, 24), y shape = (2681,)
✅ Created sequences: X shape = (9443, 30, 24), y shape = (9443,)
✅ Created sequences: X shape = (2223, 30, 24), y shape = (2223,)


INFO flwr 2026-07-19 20:13:11,398 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-19 20:13:24,113	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-19 20:13:27,838 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'GPU': 2.0, 'memory': 9315139994.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 3992202854.0, 'CPU': 4.0}
INFO flwr 2026-07-19 20:13:27,839 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-19 20:13:27,869 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-19 20:13:27,873 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-19 20:13:27,874 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-19 20:13:27,876 | server.py:91 | Evaluating initial parameters
(pid=168309) WARNING: All

  [Round 0] Test MAE: 75.3208 | NASA: 1593814.72


(DefaultActor pid=168309) I0000 00:00:1784492021.912275  168309 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(DefaultActor pid=168308) I0000 00:00:1784492032.314756  168692 cuda_dnn.cc:529] Loaded cuDNN version 91002
(pid=168308) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=168308) E0000 00:00:1784492010.451717  168308 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=168308) E0000 00:00:1784492010.466416  168308 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=168308) W0000 00:00:1784492010.498258  168308 computation_placer.cc:177] c

  [Round 1] Test MAE: 39.2932 | NASA: 125459.53


...

  [Round 49] Test MAE: 16.6159 | NASA: 7508.83


DEBUG flwr 2026-07-19 20:59:25,412 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-19 20:59:25,414 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-19 21:00:13,135 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 21:00:22,002 | server.py:125 | fit progress: (50, 0.0, {'mae': 17.553112052122138, 'nasa_score': 8218.565217012509}, 2801.5408526149986)
DEBUG flwr 2026-07-19 21:00:22,003 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 17.5531 | NASA: 8218.57


DEBUG flwr 2026-07-19 21:00:25,567 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-19 21:00:25,569 | server.py:153 | FL finished in 2805.107861954999
INFO flwr 2026-07-19 21:00:25,571 | app.py:225 | app_fit: losses_distributed [(1, 1912.912744542172), (2, 822.1624940102561), (3, 837.3472909425434), (4, 635.2388468826026), (5, 615.1120730349892), (6, 759.0021138642963), (7, 643.4610955087762), (8, 604.3699624647174), (9, 539.3319021057664), (10, 665.93471266094), (11, 503.1321944286949), (12, 523.7829989650793), (13, 508.7521495718705), (14, 556.039863412422), (15, 581.4308051259894), (16, 563.9377642982885), (17, 562.9773351401614), (18, 592.8505261404473), (19, 591.5111293257329), (20, 670.221703144542), (21, 598.5437782086824), (22, 629.7888378344084), (23, 630.2777899625008), (24, 627.0321400826438), (25, 611.7746224855122), (26, 660.7106650369209), (27, 617.1012696383292), (28, 639.8880072543495), (29, 668.8508138154682), (30, 651.13943283348

FedAvg 606: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 606, 'test_mae': 17.5531, 'nasa_score': 8218.57, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 707...")
result = run_simulation('fedavg', 'FD002', 707)
print("FedAvg 707:", result)

E0000 00:00:1784535177.121509     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784535177.177204     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784535177.608949     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784535177.608996     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784535177.609001     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784535177.609006     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 707...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8855, 30, 24), y shape = (8855,)
✅ Created sequences: X shape = (2295, 30, 24), y shape = (2295,)
✅ Created sequences: X shape = (9348, 30, 24), y shape = (9348,)
✅ Created sequences: X shape = (2060, 30, 24), y shape = (2060,)
✅ Created sequences: X shape = (9715, 30, 24), y shape = (9715,)
✅ Created sequences: X shape = (2280, 30, 24), y shape = (2280,)
✅ Created sequences: X shape = (9124, 30, 24), y shape = (9124,)
✅ Created sequences: X shape = (2542, 30, 24), y shape = (2542,)


I0000 00:00:1784535233.127020     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784535233.132972     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-20 08:13:54,895 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 08:14:02,805	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 08:14:06,379 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'memory': 21567861146.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 9243369062.0}
INFO flwr 2026-07-20 08:14:06,380 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 71.6914 | NASA: 1212383.69


(DefaultActor pid=369) I0000 00:00:1784535257.444272     369 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13632 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=368) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=368) E0000 00:00:1784535247.676192     368 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=368) E0000 00:00:1784535247.690517     368 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 19.2411 | NASA: 16577.46


DEBUG flwr 2026-07-20 08:56:17,286 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 08:56:17,287 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 08:57:25,708 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 08:57:27,097 | server.py:125 | fit progress: (50, 0.0, {'mae': 19.169558484581906, 'nasa_score': 10281.80478512135}, 2596.955413146)
DEBUG flwr 2026-07-20 08:57:27,098 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 19.1696 | NASA: 10281.80


DEBUG flwr 2026-07-20 08:57:29,453 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 08:57:29,454 | server.py:153 | FL finished in 2599.3118812770003
INFO flwr 2026-07-20 08:57:29,455 | app.py:225 | app_fit: losses_distributed [(1, 1890.0224863305725), (2, 680.7618914002802), (3, 666.9004878675651), (4, 582.8576907835145), (5, 574.6789760284074), (6, 565.4054171231833), (7, 640.4838803538547), (8, 542.4502685919324), (9, 591.9516829621289), (10, 566.2103836210878), (11, 572.5784364342625), (12, 608.7857588399817), (13, 619.0785535829212), (14, 662.0491316181078), (15, 673.1142269590523), (16, 692.7675842172091), (17, 654.4815478292155), (18, 648.5524094194917), (19, 706.7954504140467), (20, 661.7724640501134), (21, 686.4996328845424), (22, 664.8795812414975), (23, 752.5745823205368), (24, 684.5932109725103), (25, 685.184492330893), (26, 687.7705356730503), (27, 681.2543151596084), (28, 713.2355897838388), (29, 716.6679683775506), (30, 704.681431

FedAvg 707: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 707, 'test_mae': 19.1696, 'nasa_score': 10281.8, 'comm_kb': 28900.78}


In [8]:
from run_experiment import run_simulation
print("Checking FedAvg seed 808...")
result = run_simulation('fedavg', 'FD002', 808)
print("FedAvg 808:", result)

Checking FedAvg seed 808...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (9062, 30, 24), y shape = (9062,)
✅ Created sequences: X shape = (2088, 30, 24), y shape = (2088,)
✅ Created sequences: X shape = (8975, 30, 24), y shape = (8975,)
✅ Created sequences: X shape = (2433, 30, 24), y shape = (2433,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2249, 30, 24), y shape = (2249,)
✅ Created sequences: X shape = (9426, 30, 24), y shape = (9426,)
✅ Created sequences: X shape = (2240, 30, 24), y shape = (2240,)


INFO flwr 2026-07-20 09:44:41,532 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 09:44:50,360	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 09:44:54,137 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'GPU': 2.0, 'memory': 21399912039.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'object_store_memory': 9171390873.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-20 09:44:54,138 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 09:44:54,157 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 09:44:54,159 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 09:44:54,160 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 09:44:54,160 | server.py:91 | Evaluating initial parameters
(pid=106189) WARNING: Al

  [Round 0] Test MAE: 75.5367 | NASA: 1626464.11


(DefaultActor pid=106189) I0000 00:00:1784540704.407456  106189 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=106188) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=106188) E0000 00:00:1784540695.523963  106188 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=106188) E0000 00:00:1784540695.536844  106188 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=106188) W0000 00:00:1784540695.568180  106188 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 38.5244 | NASA: 47082.28


...

  [Round 49] Test MAE: 17.3141 | NASA: 8282.35


DEBUG flwr 2026-07-20 10:27:07,444 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 10:27:07,445 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 10:27:51,204 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 10:27:52,629 | server.py:125 | fit progress: (50, 0.0, {'mae': 17.640075963436406, 'nasa_score': 8313.932472118762}, 2576.2816559480007)
DEBUG flwr 2026-07-20 10:27:52,630 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 17.6401 | NASA: 8313.93


DEBUG flwr 2026-07-20 10:27:54,960 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 10:27:54,961 | server.py:153 | FL finished in 2578.6131225439995
INFO flwr 2026-07-20 10:27:54,962 | app.py:225 | app_fit: losses_distributed [(1, 1825.5657254201062), (2, 985.0340019878087), (3, 571.7986916358939), (4, 513.6956354681051), (5, 476.7220370039691), (6, 456.91472763417164), (7, 440.4301687259653), (8, 495.3013483282464), (9, 456.6645863997685), (10, 493.80364198674107), (11, 563.4193551828806), (12, 492.3662765519865), (13, 545.7266950804174), (14, 547.666259034016), (15, 553.5375127692863), (16, 529.2258749230456), (17, 581.5324308846291), (18, 586.9494319124042), (19, 519.1027013916287), (20, 531.3101771618233), (21, 523.9232076934916), (22, 548.5224607681461), (23, 557.2144600549628), (24, 564.7783578955241), (25, 533.4107198937487), (26, 558.7964584570746), (27, 560.2482124218533), (28, 564.6413941242587), (29, 548.2119204437561), (30, 545.8212

FedAvg 808: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 808, 'test_mae': 17.6401, 'nasa_score': 8313.93, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 404)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11580, 30, 24), y shape = (11580,)
✅ Created sequences: X shape = (2928, 30, 24), y shape = (2928,)
✅ Created sequences: X shape = (11863, 30, 24), y shape = (11863,)
✅ Created sequences: X shape = (2643, 30, 24), y shape = (2643,)
✅ Created sequences: X shape = (10311, 30, 24), y shape = (10311,)
✅ Created sequences: X shape = (2595, 30, 24), y shape = (2595,)
✅ Created sequences: X shape = (9514, 30, 24), y shape = (9514,)
✅ Created sequences: X shape = (2594, 30, 24), y shape = (2594,)


INFO flwr 2026-07-20 10:28:47,252 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 10:28:56,030	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 10:28:59,420 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 9155092070.0, 'accelerator_type:T4': 1.0, 'memory': 21361881498.0, 'GPU': 2.0}
INFO flwr 2026-07-20 10:28:59,421 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 10:28:59,440 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 10:28:59,440 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 10:28:59,441 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 10:28:59,442 | server.py:91 | Evaluating initial parameters
(pid=158086) WARNING: Al

  [Round 0] Test MAE: 78.5576 | NASA: 1639345.80


(DefaultActor pid=158086) I0000 00:00:1784543350.231098  158086 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=158085) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=158085) E0000 00:00:1784543340.737094  158085 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=158085) E0000 00:00:1784543340.765588  158085 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=158085) W0000 00:00:1784543340.814229  158085 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

INFO flwr 2026-07-20 10:30:52,330 | server.py:125 | fit progress: (1, 0.0, {'mae': 32.67088479380454, 'nasa_score': 78939.96550442162}, 110.71665076299905)
DEBUG flwr 2026-07-20 10:30:52,331 | server.py:173 | evaluate_round 1: strategy sampled 4 clients (out of 4)


...

  [Round 49] Test MAE: 23.1303 | NASA: 16866.48


DEBUG flwr 2026-07-20 11:18:59,701 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 11:18:59,701 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 11:20:08,212 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 11:20:09,640 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.797521837296024, 'nasa_score': 11425.869471752409}, 3068.0269597069982)
DEBUG flwr 2026-07-20 11:20:09,641 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.7975 | NASA: 11425.87


DEBUG flwr 2026-07-20 11:20:12,677 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 11:20:12,678 | server.py:153 | FL finished in 3071.064938554
INFO flwr 2026-07-20 11:20:12,679 | app.py:225 | app_fit: losses_distributed [(1, 1241.5504547232588), (2, 533.4460405456089), (3, 502.1584056386274), (4, 489.22009419154057), (5, 453.5934577204481), (6, 459.8932467974695), (7, 460.7514118293847), (8, 470.10599884827343), (9, 486.5454083410781), (10, 497.2596129463508), (11, 492.5103832230692), (12, 506.5721291609413), (13, 546.8064522073172), (14, 551.0046819637256), (15, 536.7256197791118), (16, 555.7763099188255), (17, 523.2221055580337), (18, 545.3133454191641), (19, 549.5278876719422), (20, 562.2480313779696), (21, 611.9785660016936), (22, 563.8944689045165), (23, 554.1860479475397), (24, 571.235245596166), (25, 583.0779629788877), (26, 615.4013603976225), (27, 598.5832147818072), (28, 635.9285534996968), (29, 596.4512182923497), (30, 616.94149733

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 404, 'test_mae': 22.7975, 'nasa_score': 11425.87, 'comm_kb': 28900.78}


In [10]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 505)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11230, 30, 24), y shape = (11230,)
✅ Created sequences: X shape = (3278, 30, 24), y shape = (3278,)
✅ Created sequences: X shape = (11401, 30, 24), y shape = (11401,)
✅ Created sequences: X shape = (3105, 30, 24), y shape = (3105,)
✅ Created sequences: X shape = (10389, 30, 24), y shape = (10389,)
✅ Created sequences: X shape = (2517, 30, 24), y shape = (2517,)
✅ Created sequences: X shape = (9843, 30, 24), y shape = (9843,)
✅ Created sequences: X shape = (2265, 30, 24), y shape = (2265,)


INFO flwr 2026-07-20 11:21:01,447 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 11:21:14,652	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 11:21:18,200 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'memory': 15056713728.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 6452877312.0, 'GPU': 2.0}
INFO flwr 2026-07-20 11:21:18,201 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 11:21:18,225 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 11:21:18,227 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 11:21:18,228 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 11:21:18,230 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-20 11:

  [Round 0] Test MAE: 78.2428 | NASA: 1598592.44


(pid=210520) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=210520) E0000 00:00:1784546481.011707  210520 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=210520) E0000 00:00:1784546481.040606  210520 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=210520) W0000 00:00:1784546481.281278  210520 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=210520) W0000 00:00:1784546481.281318  210520 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=210520) W0000 00:00:1784546481.281323  210520 computation_placer.cc:177] computation placer already registered. Please check linkage and avo

  [Round 1] Test MAE: 39.4839 | NASA: 250347.07


...

  [Round 49] Test MAE: 21.7063 | NASA: 30231.54


DEBUG flwr 2026-07-20 12:17:27,264 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 12:17:27,265 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 12:18:35,511 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 12:18:36,883 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.896699167067005, 'nasa_score': 82048.1157632474}, 3436.5750677429987)
DEBUG flwr 2026-07-20 12:18:36,884 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.8967 | NASA: 82048.12


DEBUG flwr 2026-07-20 12:18:40,706 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 12:18:40,707 | server.py:153 | FL finished in 3440.3987549539997
INFO flwr 2026-07-20 12:18:40,708 | app.py:225 | app_fit: losses_distributed [(1, 1873.625850588999), (2, 741.4613941074753), (3, 622.9476819432588), (4, 569.3194051972324), (5, 562.2517098535725), (6, 604.9867125836179), (7, 609.7194634963299), (8, 626.5171535302319), (9, 651.4941662307924), (10, 704.225367315201), (11, 722.8029415282656), (12, 734.1958468651163), (13, 738.3270997679078), (14, 748.2021032282978), (15, 741.5177342142805), (16, 799.7554302647783), (17, 773.0957489464671), (18, 773.9177760560258), (19, 815.6168833738157), (20, 800.091246990062), (21, 789.2000926979978), (22, 814.0590512048061), (23, 827.5235643754986), (24, 854.8232296743363), (25, 790.9848196081767), (26, 808.6506610984819), (27, 825.7313827084149), (28, 835.9641583905218), (29, 806.6899602771285), (30, 844.35445248

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 505, 'test_mae': 21.8967, 'nasa_score': 82048.12, 'comm_kb': 28900.78}


In [11]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 606)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11394, 30, 24), y shape = (11394,)
✅ Created sequences: X shape = (3114, 30, 24), y shape = (3114,)
✅ Created sequences: X shape = (11497, 30, 24), y shape = (11497,)
✅ Created sequences: X shape = (3009, 30, 24), y shape = (3009,)
✅ Created sequences: X shape = (10588, 30, 24), y shape = (10588,)
✅ Created sequences: X shape = (2318, 30, 24), y shape = (2318,)
✅ Created sequences: X shape = (9532, 30, 24), y shape = (9532,)
✅ Created sequences: X shape = (2576, 30, 24), y shape = (2576,)


INFO flwr 2026-07-20 12:19:28,208 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 12:19:41,079	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 12:19:44,867 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0, 'GPU': 2.0, 'object_store_memory': 6396840345.0, 'node:__internal_head__': 1.0, 'memory': 14925960807.0}
INFO flwr 2026-07-20 12:19:44,868 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 12:19:44,904 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 12:19:44,907 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 12:19:44,912 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 12:19:44,914 | server.py:91 | Evaluating initial parameters
(pid=266325) WARNING: Al

  [Round 0] Test MAE: 79.0803 | NASA: 1704972.01


(DefaultActor pid=266327) I0000 00:00:1784549997.249848  266327 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=266328) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=266328) E0000 00:00:1784549987.951031  266328 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=266328) E0000 00:00:1784549987.964429  266328 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=266328) W0000 00:00:1784549987.996875  266328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 39.0860 | NASA: 114083.87


...

  [Round 49] Test MAE: 22.0709 | NASA: 43282.48


DEBUG flwr 2026-07-20 13:11:23,941 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 13:11:23,943 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 13:12:33,267 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 13:12:34,649 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.921727780372866, 'nasa_score': 29881.527839846596}, 3167.609473679)
DEBUG flwr 2026-07-20 13:12:34,651 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.9217 | NASA: 29881.53


DEBUG flwr 2026-07-20 13:12:37,151 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 13:12:37,152 | server.py:153 | FL finished in 3170.1126794269985
INFO flwr 2026-07-20 13:12:37,153 | app.py:225 | app_fit: losses_distributed [(1, 1876.6100689514997), (2, 629.6168981799524), (3, 455.1485772071846), (4, 438.3374094876943), (5, 417.11022965008004), (6, 436.5369926259426), (7, 441.8986808400822), (8, 417.39410820052336), (9, 427.09680814276464), (10, 444.5338412961175), (11, 451.9933647743704), (12, 450.32604081312365), (13, 490.0049863179756), (14, 505.7788120812358), (15, 508.4085758233813), (16, 524.0942855603662), (17, 538.4079728401366), (18, 532.8679292403041), (19, 532.5696357477054), (20, 535.3826685906582), (21, 547.3084359294957), (22, 571.2652917248893), (23, 592.0194831841912), (24, 585.2248562191883), (25, 600.3910858522713), (26, 584.7538551814631), (27, 593.2789431514223), (28, 601.5304350917456), (29, 633.7104715619274), (30, 604.7

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 606, 'test_mae': 21.9217, 'nasa_score': 29881.53, 'comm_kb': 28900.78}


In [12]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 707)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11363, 30, 24), y shape = (11363,)
✅ Created sequences: X shape = (3145, 30, 24), y shape = (3145,)
✅ Created sequences: X shape = (11097, 30, 24), y shape = (11097,)
✅ Created sequences: X shape = (3409, 30, 24), y shape = (3409,)
✅ Created sequences: X shape = (10489, 30, 24), y shape = (10489,)
✅ Created sequences: X shape = (2417, 30, 24), y shape = (2417,)
✅ Created sequences: X shape = (9544, 30, 24), y shape = (9544,)
✅ Created sequences: X shape = (2564, 30, 24), y shape = (2564,)


INFO flwr 2026-07-20 13:13:25,132 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 13:13:37,222	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 13:13:40,786 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 6401662156.0, 'CPU': 4.0, 'memory': 14937211700.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0}
INFO flwr 2026-07-20 13:13:40,787 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 13:13:40,816 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 13:13:40,819 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 13:13:40,820 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 13:13:40,820 | server.py:91 | Evaluating initial parameters
(pid=319559) WARNING: Al

  [Round 0] Test MAE: 79.6442 | NASA: 1778321.99


(DefaultActor pid=319562) I0000 00:00:1784553233.473129  319562 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=319560) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=319560) E0000 00:00:1784553224.092717  319560 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=319560) E0000 00:00:1784553224.119093  319560 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=319560) W0000 00:00:1784553224.183348  319560 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 41.7863 | NASA: 20512.59


...

  [Round 49] Test MAE: 22.1989 | NASA: 94541.40


DEBUG flwr 2026-07-20 14:05:55,221 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 14:05:55,222 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 14:06:51,499 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 14:06:52,925 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.226022228117913, 'nasa_score': 92152.63373165789}, 3189.9985998949996)
DEBUG flwr 2026-07-20 14:06:52,927 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.2260 | NASA: 92152.63


DEBUG flwr 2026-07-20 14:06:56,916 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 14:06:56,917 | server.py:153 | FL finished in 3193.989858571
INFO flwr 2026-07-20 14:06:56,918 | app.py:225 | app_fit: losses_distributed [(1, 2557.203312100401), (2, 643.0630035836923), (3, 615.2879049961733), (4, 574.5621822970638), (5, 559.6261233987465), (6, 582.9624862584915), (7, 521.8753355319747), (8, 546.7308369160731), (9, 577.1337430440558), (10, 583.4823903256182), (11, 590.0119822938256), (12, 599.9742755521155), (13, 599.9964088575531), (14, 625.610043296599), (15, 621.4384552226834), (16, 620.8894048365877), (17, 660.9439851507047), (18, 626.600296291687), (19, 634.7526205379518), (20, 646.2224895740206), (21, 669.517553086564), (22, 680.1809610526387), (23, 674.0562225116917), (24, 679.4117108342137), (25, 737.2231305675068), (26, 706.6226196553628), (27, 705.2138429480505), (28, 697.3589805176416), (29, 703.3260492279358), (30, 738.6794167282574

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 707, 'test_mae': 22.226, 'nasa_score': 92152.63, 'comm_kb': 28900.78}


In [13]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 808)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11611, 30, 24), y shape = (11611,)
✅ Created sequences: X shape = (2897, 30, 24), y shape = (2897,)
✅ Created sequences: X shape = (11295, 30, 24), y shape = (11295,)
✅ Created sequences: X shape = (3211, 30, 24), y shape = (3211,)
✅ Created sequences: X shape = (10162, 30, 24), y shape = (10162,)
✅ Created sequences: X shape = (2744, 30, 24), y shape = (2744,)
✅ Created sequences: X shape = (9763, 30, 24), y shape = (9763,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)


INFO flwr 2026-07-20 14:07:46,146 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-20 14:07:58,161	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-20 14:08:01,838 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 14971643904.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 6416418816.0, 'GPU': 2.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0}
INFO flwr 2026-07-20 14:08:01,839 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-20 14:08:01,870 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-20 14:08:01,873 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-20 14:08:01,874 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-20 14:08:01,876 | server.py:91 | Evaluating initial parameters
(pid=372870) WARNING: Al

  [Round 0] Test MAE: 79.0029 | NASA: 1697344.18


(DefaultActor pid=372870) I0000 00:00:1784556495.537085  372870 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=372869) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=372869) E0000 00:00:1784556485.026913  372869 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=372872) E0000 00:00:1784556485.094545  372872 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=372869) W0000 00:00:1784556485.097932  372869 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 39.2219 | NASA: 144197.21


...

  [Round 49] Test MAE: 24.1450 | NASA: 73181.16


DEBUG flwr 2026-07-20 15:00:10,820 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-20 15:00:10,821 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-20 15:01:36,681 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 15:01:38,056 | server.py:125 | fit progress: (50, 0.0, {'mae': 24.560671360261978, 'nasa_score': 95467.79005061771}, 3214.0593351569987)
DEBUG flwr 2026-07-20 15:01:38,057 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 24.5607 | NASA: 95467.79


DEBUG flwr 2026-07-20 15:01:41,844 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-20 15:01:41,845 | server.py:153 | FL finished in 3217.848631323999
INFO flwr 2026-07-20 15:01:41,846 | app.py:225 | app_fit: losses_distributed [(1, 1875.619459183054), (2, 582.6404828072191), (3, 567.0368648255479), (4, 613.654685966626), (5, 637.7381959877515), (6, 655.4596037643663), (7, 688.4662768633183), (8, 754.2519680117564), (9, 773.3018161166914), (10, 751.9320946465414), (11, 757.6077233658866), (12, 765.2570716998955), (13, 749.719701700363), (14, 797.4697979110005), (15, 805.1219695120973), (16, 789.9427047937994), (17, 783.0527784629114), (18, 807.4549726149094), (19, 803.6611913510856), (20, 805.8633357929824), (21, 835.942298534162), (22, 802.3223837705726), (23, 809.6706453789939), (24, 824.3458808721086), (25, 843.923412287907), (26, 828.6938661352336), (27, 844.2057305257896), (28, 834.0592994618397), (29, 837.6785353380963), (30, 842.37770100053

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 808, 'test_mae': 24.5607, 'nasa_score': 95467.79, 'comm_kb': 28900.78}
